# Bài 21 · Khuếch tán & flow matching: học khử nhiễu, rồi biến nhiễu thành chữ số mới

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/21-diffusion.ipynb)

Notebook đi kèm [Bài 21 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/diffusion/) (demo tương tác nằm ở trang bài học).
Ta tự viết bằng numpy một mô hình **flow matching**: thêm nhiễu vào dữ liệu theo một công thức, dạy một mạng nhỏ đoán
hướng khử nhiễu, rồi đi ngược từ nhiễu thuần về dữ liệu. Đầu tiên trên một vòng 8 cụm điểm 2 chiều, nơi bộ khử nhiễu
tốt nhất tính được chính xác; sau đó trên mã 16 số của autoencoder Bài 19, đúng cách Stable Diffusion làm.

**Bạn sẽ làm:**
1. Thêm nhiễu bằng một công thức đóng $x_t = (1 - t)\,x_0 + t\,\varepsilon$, và tính bộ khử nhiễu tối ưu của vòng
   8 cụm: mất mát của nó không bằng 0.
2. Viết tay lan truyền ngược của một MLP dư, huấn luyện nó bằng flow matching, rồi sinh mẫu bằng các bước Euler.
3. Kiểm chứng rằng đoán nhiễu, đoán ảnh và đoán vận tốc đổi được cho nhau, và một bước DDIM chính là một bước Euler:
   số bước là lựa chọn của bộ giải, không phải của mô hình. Rồi làm thẳng đường đi bằng reflow để một bước là đủ.
4. Chạy mạng flow matching và bộ giải mã mà demo dùng để sinh chữ số MNIST, và đọc kết quả quét đầy đủ (FD, recall,
   sao chép).
5. Thấy bộ khử nhiễu tối ưu chỉ chép lại ảnh huấn luyện, và hướng dẫn không cần bộ phân loại đổi đa dạng lấy độ bám.

**Cần biết trước:** [Bài 19 · Autoencoder & VAE](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/)
(bộ giải mã và mã 16 chiều dùng lại ở đây), [Bài 20 · GAN](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gan/)
(thước đo FD, precision/recall, phép thử sao chép), khối dư của
[Bài 13 · ResNet](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/), Adam của
[Bài 07](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả
tải ~10 MB ảnh MNIST huấn luyện và ~1 MB kết quả của demo lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng các lần huấn luyện và
đánh giá đầy đủ (20.000 bước × 3 seed cho mỗi cấu hình, 10.000 mẫu mỗi ô, khoảng một giờ trên 12 nhân) được nạp từ
đúng các tệp mà demo dùng; notebook chạy lại một phiên bản nhỏ của từng cơ chế và kiểm tra những gì tính lại được.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import math
import time
import urllib.request
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng điểm, nét, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”). Làm tròn nửa lên trên dạng thập phân ngắn nhất của số,
    như trang web (Intl.NumberFormat): 3,55 → 3,6, trong khi round(3.55, 1) của Python cho 3,5."""
    q = Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-digits), ROUND_HALF_UP)
    s = f"{float(q) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def span(values, digits=1, scale=1.0, unit=""):
    """Khoảng min–max qua các seed, như demo: 3,4–3,6 (một số khi hai đầu làm tròn như nhau)."""
    lo, hi = scale * min(values), scale * max(values)
    if vn(lo, digits) == vn(hi, digits):
        return vn(lo, digits) + unit
    return f"{vn(lo, digits)}–{vn(hi, digits)}{unit}"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 1,9·10⁻⁷."""
    e = math.floor(math.log10(abs(x)))
    return f"{vn(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def tile(images, rows, cols, gap=2):
    """Ghép rows × cols ảnh 28×28 thành một ảnh lớn, nền trắng giữa các ô."""
    out = np.zeros((rows * (28 + gap) - gap, cols * (28 + gap) - gap))
    for i, im in enumerate(images[: rows * cols]):
        r, c = divmod(i, cols)
        out[r * (28 + gap) : r * (28 + gap) + 28, c * (28 + gap) : c * (28 + gap) + 28] = np.reshape(im, (28, 28))
    return out


def show_tile(ax, images, rows, cols, title=None):
    ax.imshow(tile(images, rows, cols), cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527), span([3.55, 3.44, 3.35]), sci(4.04e-5))

## 2 · Dữ liệu

Vòng 8 cụm 2 chiều (mục 3–9) tự tạo dữ liệu. Phần MNIST cần:
- **ảnh MNIST huấn luyện** (LeCun và cộng sự, 1998): 60.000 ảnh xám 28×28 cùng nhãn. Ô dưới chỉ tải hai tệp huấn luyện
  và kiểm tra mã MD5; ta dùng 1.000 ảnh đầu tiên;
- năm tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5):
  - `diffusion-net.json`: hai mạng flow matching chạy trong trình duyệt (H = 256, 40.000 bước, seed 0; một mạng có
    điều kiện nhãn), ma trận lưu bằng int8 với một hệ số float16 cho mỗi cột;
  - `diffusion-dec.json`: bộ giải mã 16 → 128 → 256 → 784 của autoencoder sâu Bài 19 (seed 0), cùng cách nén;
  - `diffusion-mnist.json`: kết quả quét đầy đủ (mọi bộ giải × số bước, hướng dẫn, các mốc so sánh, tỉ lệ sao chép), và
    mã của 1.000 ảnh huấn luyện đầu tiên;
  - `diffusion-2d.json`, `diffusion-reflow.json`: bảng của vòng 8 cụm và mạng reflow 2 chiều.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST (chỉ hai tệp huấn luyện)
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


# Kết quả của các lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
NETS = fetch_json("diffusion-net.json", "cc13e79862e744dc54a7d33815298605")
DEC = fetch_json("diffusion-dec.json", "b4e6a6cc2b2748aeeef7aaa681f919b9")
MN = fetch_json("diffusion-mnist.json", "cea01777115494363a7d2151b16bbc1a")
RING2D = fetch_json("diffusion-2d.json", "d14d58d79a9f2f268e92fc31a677974b")
REFLOW = fetch_json("diffusion-reflow.json", "c1b5b583aaa2ddc9b699fb735ef60a21")
names = [
    "diffusion-net.json",
    "diffusion-dec.json",
    "diffusion-mnist.json",
    "diffusion-2d.json",
    "diffusion-reflow.json",
]
print(f"5 tệp của demo: {vn(sum((Path('data') / f).stat().st_size for f in names) / 1e6, 1)} MB")

Ma trận của hai mạng flow matching là số nguyên int8 nhân với một hệ số float16 cho mỗi cột, độ lệch là float16; bộ giải
mã nén như Bài 19 (lớp ẩn float16, ma trận lớp ra int8). Ô dưới giải nén chúng thành float64, đọc 1.000 ảnh MNIST đầu tiên và mã 16 số của chúng.

Mã ở đây đã được **chia cho độ lệch chuẩn** của mọi mã huấn luyện (`scale` = 1/độ lệch chuẩn), nên mỗi số có thang
đo khoảng 1, cùng thang với nhiễu $\varepsilon \sim \mathcal N(0, I)$. Stable Diffusion làm đúng việc này với một hệ số
riêng của nó (Bài 19). Muốn giải mã thì làm ngược lại: chia mã cho `scale`.

In [ ]:
def b64(s, dtype):
    """Giải mã một mảng mà demo lưu dạng base64 (little-endian)."""
    return np.frombuffer(base64.b64decode(s), dtype)


def unpack(t):
    if "f16" in t:
        return b64(t["f16"], "<f2").astype(np.float64).reshape(t["shape"])
    if "f32" in t:
        return b64(t["f32"], "<f4").astype(np.float64).reshape(t["shape"])
    scale = b64(t["scale"], "<f2").astype(np.float64)  # một hệ số cho mỗi cột
    return b64(t["i8"], np.int8).astype(np.float64).reshape(t["shape"]) * scale


with gzip.open(DATA / "train-images-idx3-ubyte.gz") as f:
    X_img = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)[:1000] / 255
with gzip.open(DATA / "train-labels-idx1-ubyte.gz") as f:
    y_all = np.frombuffer(f.read(), np.uint8, offset=8)
y_img = y_all[:1000].astype(int)
print(f"MNIST huấn luyện: {vn(len(y_all), 0)} ảnh; ta dùng {vn(len(X_img), 0)} ảnh đầu tiên")

SCALE = NETS["scale"]
dec = {k: unpack(t) for k, t in DEC["dec"].items()}
Zc = b64(MN["memo"]["codes"], "<f2").astype(np.float64).reshape(-1, 16)  # mã (đã chia) của 1.000 ảnh đầu tiên


def decode(Z):
    """Mã đã chia → ảnh E[x|z]: bộ giải mã 16 → 128 → 256 → 784 của Bài 19 (ReLU, sigmoid ở lớp ra)."""
    h = np.maximum(np.atleast_2d(Z) / SCALE @ dec["d0.W"] + dec["d0.b"], 0)
    h = np.maximum(h @ dec["d1.W"] + dec["d1.b"], 0)
    return 0.5 * (1 + np.tanh(0.5 * (h @ dec["out.W"] + dec["out.b"])))


def nearest(A, B):
    """Khoảng cách từ mỗi hàng của A tới hàng gần nhất của B, và chỉ số hàng đó."""
    d2 = np.maximum((A**2).sum(1)[:, None] - 2 * A @ B.T + (B**2).sum(1)[None], 0)
    i = d2.argmin(1)
    return np.sqrt(d2[np.arange(len(A)), i]), i


R_img = decode(Zc)
own = (nearest(R_img, X_img)[1] == np.arange(1000)).mean()
print(f"mã: {Zc.shape}, độ lệch chuẩn {vn(Zc.std(), 3)} · scale = {vn(SCALE, 4)}")
print(f"ảnh dựng lại từ mã gần ảnh gốc của chính nó nhất trong {pct(own, 1)} trường hợp; "
      f"sai số bình phương trung bình {vn(((R_img - X_img) ** 2).sum(1).mean())} mỗi ảnh")  # fmt: skip
assert own > 0.99, "mã không khớp với 1.000 ảnh huấn luyện đầu tiên: hãy kiểm tra lại dữ liệu"

fig, axes = plt.subplots(2, 1, figsize=(10, 2.5))
show_tile(axes[0], X_img[:12], 1, 12, "ảnh MNIST huấn luyện")
show_tile(axes[1], R_img[:12], 1, 12, "giải mã từ 16 số của mỗi ảnh (bộ giải mã Bài 19)")
plt.tight_layout()
plt.show()

Hàng trên là ảnh gốc, hàng dưới là ảnh giải mã từ mã 16 số của chúng: hơi mờ nhưng đúng chữ số và dáng viết. Mỗi ảnh
dựng lại gần ảnh gốc của nó hơn mọi ảnh khác trong 1.000 ảnh, nên mã đúng là của các ảnh này.

Từ mục 10, mô hình khuếch tán chỉ làm việc với các mã 16 số; bộ giải mã cố định vẽ chúng thành ảnh. Đó là
**khuếch tán trong không gian tiềm ẩn** (latent diffusion) của Stable Diffusion, thu nhỏ.

## 3 · Quá trình thuận: trộn dữ liệu với nhiễu

Dữ liệu đồ chơi là vòng 8 cụm của demo: 8 cụm Gauss đặt đều trên một vòng tròn bán kính 2, mỗi cụm có độ lệch chuẩn
$s = 0{,}1$. Quá trình thuận của flow matching (theo quy ước của Stable Diffusion 3) trộn một điểm sạch $x_0$ với nhiễu
$\varepsilon \sim \mathcal N(0, I)$:

$$x_t = (1 - t)\,x_0 + t\,\varepsilon, \qquad t \in [0, 1].$$

$t = 0$ là dữ liệu sạch, $t = 1$ là nhiễu thuần. Ở đây không có gì phải học, và cũng không phải mô phỏng từng bước:
muốn lấy điểm ở mức nhiễu nào, ta tính thẳng bằng công thức. Hình dưới vẽ cùng 2.000 điểm ở năm mức $t$ (ô vuông cam)
và tâm của 8 cụm (vòng tròn xanh).

In [ ]:
K, RADIUS, S_RING = 8, 2.0, 0.1
MEANS = RADIUS * np.array([[np.cos(2 * np.pi * k / K), np.sin(2 * np.pi * k / K)] for k in range(K)])


def ring_data(n, s=S_RING, seed=0):
    rng = np.random.default_rng(seed)
    return MEANS[rng.integers(0, K, n)] + s * rng.standard_normal((n, 2))


x0_demo = ring_data(2000)
eps_demo = np.random.default_rng(1).standard_normal((2000, 2))
T_SHOW = [0, 0.25, 0.5, 0.75, 1]
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5), sharex=True, sharey=True)
for ax, t in zip(axes, T_SHOW):
    xt = (1 - t) * x0_demo + t * eps_demo
    ax.scatter(*xt.T, s=2, marker="s", color=ORANGE, alpha=0.5, lw=0)
    ax.scatter(*MEANS.T, s=45, facecolors="none", edgecolors=BLUE, lw=1.5)
    ax.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), aspect="equal", title=f"t = {vn(t, 2)}")
plt.tight_layout()
vn_axes(fig)
plt.show()
sd_t = [((1 - t) * x0_demo + t * eps_demo).std(0).mean() for t in T_SHOW]
print("độ lệch chuẩn của xₜ theo mỗi trục:", " · ".join(f"t = {vn(t, 2)}: {vn(v)}" for t, v in zip(T_SHOW, sd_t)))

Đến $t = 0{,}5$ các cụm đã hòa vào nhau; ở $t = 1$ không còn dấu vết nào của dữ liệu. Một chi tiết: đám mây ở giữa đường
**hẹp hơn** ở hai đầu (độ lệch chuẩn 0,84 ở $t = 0{,}75$, so với 1,41 ở $t = 0$ và 1,00 ở $t = 1$), vì phương sai là
$(1 - t)^2 \operatorname{Var}(x_0) + t^2$. Đường của flow matching là đoạn thẳng giữa hai điểm; nó không giữ phương sai
không đổi như lịch của DDPM (mục 8).

## 4 · Bộ khử nhiễu tốt nhất, tính chính xác

Mạng sẽ học **vận tốc** $v = \varepsilon - x_0$ (đạo hàm của $x_t$ theo $t$) chỉ từ $x_t$ và $t$. Nhưng nhìn $x_t$, ta
không biết nó đến từ cặp $(x_0, \varepsilon)$ nào. Dự đoán tốt nhất theo sai số bình phương là trung bình có điều kiện:

$$v^*(x_t, t) = \mathbb E[\varepsilon - x_0 \mid x_t] = \frac{x_t - \mathbb E[x_0 \mid x_t]}{t}.$$

Với hỗn hợp Gauss, $\mathbb E[x_0 \mid x_t]$ tính được chính xác. Nếu $x_0$ thuộc cụm $k$ (tâm $m_k$), thì
$x_t \sim \mathcal N\big((1-t)\,m_k,\ \sigma_t^2 I\big)$ với $\sigma_t^2 = (1-t)^2 s^2 + t^2$. Xác suất hậu nghiệm của
mỗi cụm là một softmax của $-\lVert x_t - (1-t)m_k \rVert^2 / 2\sigma_t^2$, và trong cụm đó
$\mathbb E[x_0 \mid x_t, k] = m_k + \frac{(1-t)s^2}{\sigma_t^2}\big(x_t - (1-t)m_k\big)$. Hàm dưới viết cho dạng tổng
quát $x_t = a\,x_0 + b\,\varepsilon$, để mục 8 dùng lại được với lịch của DDPM.

In [ ]:
def ring_x0(x, a, b, s=S_RING):
    """E[x₀ | xₜ] cho xₜ = a·x₀ + b·ε, x₀ ~ (1/K) Σ N(m_k, s² I): trung bình các tâm theo trọng số hậu nghiệm.
    a, b là số hoặc mảng (một giá trị cho mỗi điểm)."""
    a = np.broadcast_to(np.asarray(a, float), (len(x),))[:, None, None]
    b = np.broadcast_to(np.asarray(b, float), (len(x),))[:, None, None]
    var = a * a * s * s + b * b  # phương sai của xₜ trong mỗi cụm
    diff = x[:, None, :] - a * MEANS[None]  # (n, K, 2)
    lw = -(diff**2).sum(-1) / (2 * var[:, :, 0])
    w = np.exp(lw - lw.max(1, keepdims=True))
    w /= w.sum(1, keepdims=True)  # xác suất hậu nghiệm của mỗi cụm
    return (w[:, :, None] * (MEANS[None] + (a * s * s / var) * diff)).sum(1)


def ring_v(x, t, s=S_RING):
    """Vận tốc tối ưu v*(xₜ, t) = (xₜ − E[x₀|xₜ]) / t của flow matching (t là số hoặc mảng)."""
    tc = np.broadcast_to(np.asarray(t, float), (len(x),))[:, None]
    return (x - ring_x0(x, 1 - tc[:, 0], tc[:, 0], s)) / tc

### 🤔 Dự đoán trước
Mạng hoàn hảo, tức đúng bằng $v^*$, đạt mất mát flow matching $\mathbb E\,\lVert v^*(x_t, t) - (\varepsilon - x_0)\rVert^2$
(với $t$ đều trên $[0, 1]$) bằng bao nhiêu? Có bằng 0 không?

In [ ]:
rng = np.random.default_rng(2)
n_mc = 200_000
x0_mc, e_mc, t_mc = ring_data(n_mc, seed=3), rng.standard_normal((n_mc, 2)), rng.random(n_mc)
xt_mc = (1 - t_mc[:, None]) * x0_mc + t_mc[:, None] * e_mc
target = e_mc - x0_mc
floor = ((ring_v(xt_mc, t_mc) - target) ** 2).sum(1).mean()
print(f"mất mát flow matching của bộ khử nhiễu tối ưu: {vn(floor, 2)} (trung bình trên {vn(n_mc, 0)} bộ (x₀, ε, t))")
print(f"nếu đoán một hằng số cho mọi điểm (trung bình của ε − x₀): {vn(target.var(0).sum(), 2)}")
assert floor > 3, "bộ khử nhiễu tối ưu không thể có mất mát gần 0"

Không bằng 0, mà khoảng 3,6: phần lớn của mục tiêu $\varepsilon - x_0$ là **không đoán được** từ $x_t$. Mỗi điểm $x_t$
có thể đến từ rất nhiều cặp (điểm dữ liệu, nhiễu); mạng chỉ học được trung bình của chúng. Vì thế mất mát của khuếch
tán không bao giờ về 0, và không dùng được giá trị mất mát để so hai mô hình khác cách đặt $t$.

Phần đoán được phụ thuộc vào $t$. Ô dưới đo, ở từng $t$, tỉ lệ phương sai của mục tiêu mà $v^*$ giải thích được
(đường xanh, chấm tròn), và so với 8 giá trị mà demo tính bằng JavaScript (ô vuông cam).

In [ ]:
T_GRID = np.round(np.arange(0.05, 1.0, 0.05), 2)
x0_s, e_s, g_s = x0_mc[:20000], e_mc[:20000], target[:20000]
share = np.array(
    [1 - ((ring_v((1 - t) * x0_s + t * e_s, t) - g_s) ** 2).sum() / ((g_s - g_s.mean(0)) ** 2).sum() for t in T_GRID]
)
demo_share = {e["t"]: e["share"] for e in RING2D["table"]["0.1"]["explained"]}
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(T_GRID, 100 * share, color=BLUE, marker="o", ms=4, label="notebook (20.000 điểm mỗi t)")
ax.scatter(list(demo_share), [100 * v for v in demo_share.values()], marker="s", color=ORANGE, zorder=3, label="demo")
ax.set(xlabel="t", ylabel="phương sai giải thích được (%)", ylim=(0, 100), title="v* đoán được bao nhiêu của ε − x₀?")
ax.legend(frameon=False)
vn_axes(fig)
plt.show()
print("  ".join(f"t = {vn(t, 2)}: {pct(s, 0)}" for t, s in zip(T_GRID, share) if t in demo_share))
assert abs(share[T_GRID == 0.5][0] - demo_share[0.5]) < 0.02, "khác với demo: hãy kiểm tra lại bộ khử nhiễu"

Gần $t = 0$, $x_t$ gần như là dữ liệu, nên biết $x_t$ là biết $x_0$ thuộc cụm nào: $v^*$ giải thích phần lớn mục tiêu.
Ở $t = 0{,}5$ chỉ còn 26%, và quanh $t = 0{,}7$ gần như 0: các cụm đã hòa vào nhau, và vận tốc tối ưu chỉ là một
trung bình mờ của 8 hướng. Gần $t = 1$, $x_t \approx \varepsilon$ nên phần $\varepsilon$ của mục tiêu lại đoán được.

## 5 · Sinh mẫu: đi ngược từ nhiễu bằng các bước Euler

Sinh mẫu là giải phương trình vi phân $\frac{dx}{dt} = v(x, t)$ ngược từ $t = 1$ (nhiễu thuần) về $t = 0$. Cách đơn giản
nhất là các bước **Euler** cỡ $\Delta t$:

$$x_{t - \Delta t} = x_t - \Delta t\; v(x_t, t).$$

Mỗi bước là một lần gọi mạng. Để so được với demo, ta lấy đúng các nhiễu mà demo dùng: bộ sinh số ngẫu nhiên
`mulberry32` của trình duyệt (viết lại bằng numpy, trùng từng bit như ở Bài 09) và biến đổi Box–Muller.
Độ đo của demo: tỉ lệ mẫu nằm trong một cụm (cách tâm gần nhất dưới $3s$), số cụm có ít nhất 1% số mẫu, và bán kính
trung bình (dữ liệu: 2).

In [ ]:
def mulberry32(seed, n):
    """n số nguyên 32 bit đầu tiên của mulberry32(seed), bộ sinh mà demo (JavaScript) dùng, viết dạng vectơ."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)  # phép nhân uint32 tự quay vòng mod 2³², như Math.imul
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return t ^ (t >> 14)


def js_normals(seed, n):
    """normals(mulberry32(seed), n) của demo: Box–Muller trên từng cặp số đều."""
    u = mulberry32(seed, n) / 2**32
    r, a = np.sqrt(-2 * np.log(1 - u[0::2])), 2 * np.pi * u[1::2]
    return np.stack([r * np.cos(a), r * np.sin(a)], 1).ravel()


def grid(k):
    return np.linspace(1, 0, k + 1)  # k bước đều theo t, từ 1 về 0


def euler(v, z, ts, path=None):
    x = z.copy()
    for a, b in itertools.pairwise(ts):
        x = x - (a - b) * v(x, a)
        if path is not None:
            path.append(x)
    return x


def ring_quality(x, s=S_RING):
    d = np.sqrt(((x[:, None] - MEANS[None]) ** 2).sum(-1))
    near = d.min(1) < 3 * s
    counts = np.bincount(d.argmin(1)[near], minlength=K)
    return near.mean(), int((counts >= 0.01 * len(x)).sum()), np.hypot(*x.T).mean()


Z_DEMO = [js_normals(seed, 4000).reshape(-1, 2) for seed in (1, 2, 3)]  # 3 seed × 2.000 điểm nhiễu của demo
print("4 số đều đầu tiên của mulberry32(1):", [vn(u, 6) for u in mulberry32(1, 4) / 2**32])
assert mulberry32(1, 1)[0] / 2**32 == 0.6270739405881613  # giá trị mà reel/diffusion/diff.py kiểm tra với JavaScript

### 🤔 Dự đoán trước
Dùng chính bộ khử nhiễu tối ưu (không có sai số học nào) và chỉ **một** bước Euler từ $t = 1$ về $t = 0$.
2.000 điểm nhiễu sẽ rơi vào đâu? Bao nhiêu phần trăm nằm trong một cụm?

In [ ]:
x_one = euler(ring_v, Z_DEMO[0], grid(1))
q_one = ring_quality(x_one)
print(f"sau 1 bước: {pct(q_one[0], 0)} mẫu trong một cụm, {q_one[1]}/8 cụm có mẫu, "
      f"điểm xa gốc nhất cách gốc {vn(np.hypot(*x_one.T).max(), 6)}")  # fmt: skip
print("trung bình của dữ liệu:", [vn(m, 6) for m in MEANS.mean(0)])
assert q_one[0] == 0 and np.hypot(*x_one.T).max() < 1e-9

Mọi điểm rơi đúng vào gốc tọa độ, trung bình của dữ liệu, nơi không có cụm nào. Ở $t = 1$, $x_t$ là nhiễu thuần nên
không chứa thông tin gì về $x_0$: dự đoán tốt nhất $\mathbb E[x_0 \mid x_1]$ là trung bình của **mọi** điểm dữ liệu,
và một bước Euler đi thẳng tới đó. Từng cặp (nhiễu, dữ liệu) được nối bằng một đoạn thẳng, nhưng trường vận tốc tối ưu
là trung bình của mọi đoạn thẳng đi qua $x_t$, và trung bình đó **cong**. Muốn đi theo đường cong thì phải đi nhiều bước.

Ô dưới chạy 1, 2, 4, 8, 32 và 100 bước Euler trên 3 bộ nhiễu của demo, và so với bảng mà demo tính sẵn.

In [ ]:
STEPS = [1, 2, 4, 8, 32, 100]
table = RING2D["table"]["0.1"]["rows"]
for k in STEPS:
    qs = [ring_quality(euler(ring_v, z, grid(k))) for z in Z_DEMO]
    stored = [r["inMode"] for r in table[f"euler/{k}"]]
    print(f"{k:>5} bước: trong một cụm {span([q[0] for q in qs], 1, 100, '%'):>12}, "
          f"{span([q[1] for q in qs], 0)}/8 cụm, bán kính {span([q[2] for q in qs], 2)} "
          f"· demo: {span(stored, 1, 100, '%')}")  # fmt: skip
    assert np.allclose([q[0] for q in qs], stored, atol=1e-3), "khác với demo: hãy kiểm tra lại bộ giải"

Bộ khử nhiễu là tối ưu, nên mọi sai số ở đây là của **bộ giải**. 2 bước chỉ đưa 4,2–4,6% mẫu vào cụm, 4 bước đã được
76,8–78,4%, 8 bước 97,3–98,2%, rồi tăng thêm số bước không còn đổi gì đáng kể. Hình dưới vẽ đường đi của 48 điểm nhiễu
với 4 và 32 bước (xám), điểm cuối (ô vuông cam) và tâm cụm (vòng tròn xanh). Với 32 bước, đường đi cong rõ: đầu tiên mọi điểm cùng kéo về
gốc, đến giữa đường mới tách ra về từng cụm.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, k in zip(axes, (4, 32)):
    path = [Z_DEMO[0]]
    xk = euler(ring_v, Z_DEMO[0], grid(k), path)
    P = np.array(path)[:, :48]  # (bước, điểm, 2)
    for j in range(48):
        ax.plot(P[:, j, 0], P[:, j, 1], color=GREY, lw=0.7, alpha=0.7)
    ax.scatter(*xk.T, s=3, marker="s", color=ORANGE, alpha=0.6, lw=0)
    ax.scatter(*MEANS.T, s=60, facecolors="none", edgecolors=BLUE, lw=1.5)
    ax.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), aspect="equal", xlabel="x₁")
    ax.set_title(f"{k} bước Euler: {pct(ring_quality(xk)[0], 0)} mẫu trong một cụm", fontsize=10)
axes[0].set_ylabel("x₂")
plt.tight_layout()
vn_axes(fig)
plt.show()

z_path = js_normals(4, 1000).reshape(-1, 2)  # 500 điểm, như demo
path = [z_path]
x_end = euler(ring_v, z_path, grid(1000), path)
length = np.hypot(*np.diff(np.array(path), axis=0).transpose(2, 0, 1)).sum(0)
chord = np.hypot(*(x_end - z_path).T)
ratios = np.sort(length[chord > 1e-9] / chord[chord > 1e-9])
ratio = ratios[len(ratios) // 2]  # trung vị theo cách của demo (phần tử giữa của dãy đã sắp xếp)
print(f"đường đi thật (1.000 bước) dài gấp {vn(ratio, 2)} lần đoạn thẳng nối điểm đầu và điểm cuối (trung vị)")
assert abs(ratio - RING2D["table"]["0.1"]["pathRatio"]) < 1e-3, "khác với demo"

Đường đi trung vị dài gấp 1,5 lần dây cung. Chỗ cong này là lý do một bước không đủ (mục 9 làm thẳng nó lại).

## 6 · Huấn luyện một mạng flow matching

Bộ khử nhiễu chính xác chỉ có được khi biết phân phối dữ liệu. Thông thường ta chỉ có các mẫu, nên ta học $v_\theta$
bằng hồi quy:

$$\mathcal L = \mathbb E_{x_0,\ \varepsilon,\ t \sim \mathcal U(0, 1)}\ \big\lVert v_\theta(x_t, t) - (\varepsilon - x_0) \big\rVert^2.$$

Mạng giống hệt mạng của demo (`reel/diffusion/diff.py`): một MLP dư theo quy ước hàng $h = f(xW + b)$. Đầu vào là $x_t$
nối với 17 đặc trưng của $t$ (chính $t$, và $\sin$, $\cos$ của $2^k \pi t$ với $k < 8$), rồi
$h_1 = \mathrm{SiLU}(x W_0 + b_0)$, hai khối dư $h \leftarrow h + \mathrm{SiLU}(h W_i + b_i)$ (Bài 13), và lớp ra tuyến tính
$W_o$ khởi tạo bằng 0. $\mathrm{SiLU}(a) = a\,\sigma(a)$, với đạo hàm $\sigma(a)\,(1 + a(1 - \sigma(a)))$.
Mạng có điều kiện (mục 12) nối thêm một vectơ one-hot 11 chiều cho nhãn (10 chữ số và "không nhãn" ∅).

In [ ]:
def temb(t, n):
    """17 đặc trưng của t: t, sin(2ᵏπt), cos(2ᵏπt) với k = 0…7."""
    t = np.broadcast_to(np.asarray(t, float), (n,))[:, None]
    a = t * (2.0 ** np.arange(8) * np.pi)[None]
    return np.concatenate([t, np.sin(a), np.cos(a)], 1)


def inputs(x, t, y=None):
    parts = [x, temb(t, len(x))]
    if y is not None:  # nhãn 0–9, hoặc 10 = ∅
        parts.append(np.eye(11)[y])
    return np.concatenate(parts, 1)


def sigmoid(a):
    return 0.5 * (1 + np.tanh(0.5 * a))


def init(din, dout, H, seed, L=3):
    rng = np.random.default_rng(seed)
    p = {"W0": rng.standard_normal((din, H)) * np.sqrt(2 / din), "b0": np.zeros(H)}
    for i in range(1, L):
        p[f"W{i}"], p[f"b{i}"] = rng.standard_normal((H, H)) * np.sqrt(1 / H), np.zeros(H)
    p["Wo"], p["bo"] = np.zeros((H, dout)), np.zeros(dout)
    return p


def forward(p, X, keep=False):
    a = X @ p["W0"] + p["b0"]
    s = sigmoid(a)
    h, cache = a * s, [(X, a, s)]
    for i in range(1, sum(k[0] == "W" for k in p) - 1):  # các khối dư W1, W2, …
        a = h @ p[f"W{i}"] + p[f"b{i}"]
        s = sigmoid(a)
        cache.append((h, a, s))
        h = h + a * s
    out = h @ p["Wo"] + p["bo"]
    return (out, (cache, h)) if keep else out


def backward(p, cache, dout):
    cache, h = cache
    g = {"Wo": h.T @ dout, "bo": dout.sum(0)}
    dh = dout @ p["Wo"].T
    for i in range(len(cache) - 1, -1, -1):
        hin, a, s = cache[i]
        da = dh * s * (1 + a * (1 - s))  # đạo hàm của SiLU
        g[f"W{i}"], g[f"b{i}"] = hin.T @ da, da.sum(0)
        dh = (dh if i > 0 else 0) + da @ p[f"W{i}"].T  # khối dư: gradient đi qua cả đường tắt
    return g

Trước khi tin lan truyền ngược viết tay, ta so nó với sai phân hữu hạn trên một mạng nhỏ có trọng số ngẫu nhiên
(cả $W_o$ khác 0, để gradient chạy qua mọi lớp):

In [ ]:
rng = np.random.default_rng(0)
p_chk = {k: v + 0.3 * rng.standard_normal(v.shape) for k, v in init(5, 3, 6, 1).items()}
X_chk, tgt_chk = rng.standard_normal((4, 5)), rng.standard_normal((4, 3))
out, cache = forward(p_chk, X_chk, keep=True)
g = backward(p_chk, cache, out - tgt_chk)  # gradient của ½‖out − tgt‖²
worst = 0.0
for k, w in p_chk.items():
    for i in range(w.size):
        old = w.flat[i]
        w.flat[i] = old + 1e-6
        lp = 0.5 * ((forward(p_chk, X_chk) - tgt_chk) ** 2).sum()
        w.flat[i] = old - 1e-6
        lm = 0.5 * ((forward(p_chk, X_chk) - tgt_chk) ** 2).sum()
        w.flat[i] = old
        num = (lp - lm) / 2e-6
        worst = max(worst, abs(num - g[k].flat[i]) / max(1e-8, abs(num) + abs(g[k].flat[i])))
print(f"{sum(w.size for w in p_chk.values())} tham số, sai lệch tương đối lớn nhất {sci(worst)}")
assert worst < 1e-6

Lan truyền ngược khớp sai phân hữu hạn. Vòng huấn luyện dưới giống demo: mỗi bước lấy 256 điểm dữ liệu, 256 nhiễu mới
và 256 giá trị $t$ đều, tính $x_t$ bằng công thức của mục 3, rồi một bước Adam (Bài 07) trên sai số bình phương với
mục tiêu $\varepsilon - x_0$. Tốc độ học tăng dần trong 2% số bước đầu rồi giảm theo cosin. Mô hình dùng để sinh mẫu là
**trung bình trượt (EMA)** của trọng số, như mọi mô hình khuếch tán thật. Truyền `E` khi muốn ghép sẵn mỗi điểm dữ liệu
với một nhiễu cố định (mục 9).

In [ ]:
def lr_at(s, lr, steps):
    warm = max(1, steps // 50)
    return lr * (s + 1) / warm if s < warm else lr * 0.5 * (1 + np.cos(np.pi * (s - warm) / (steps - warm)))


def train_fm(X, seed=0, E=None, H=64, steps=3000, lr=5e-3, batch=256, ema_decay=0.995):
    """Flow matching trên các hàng của X: trả về trọng số EMA và mất mát của từng bước."""
    rng = np.random.default_rng(seed)
    d = X.shape[1]
    p = init(d + 17, d, H, seed)
    m1, m2 = {k: np.zeros_like(v) for k, v in p.items()}, {k: np.zeros_like(v) for k, v in p.items()}
    ema, losses = {k: v.copy() for k, v in p.items()}, []
    for s in range(steps):
        idx = rng.integers(0, len(X), batch)
        x0, t = X[idx], rng.random(batch)
        e = E[idx] if E is not None else rng.standard_normal((batch, d))
        xt = (1 - t[:, None]) * x0 + t[:, None] * e
        out, cache = forward(p, inputs(xt, t), keep=True)
        diff = out - (e - x0)
        losses.append((diff**2).sum(1).mean())
        g = backward(p, cache, (2 / diff.size) * diff)
        c1, c2, lr_s = 1 - 0.9 ** (s + 1), 1 - 0.999 ** (s + 1), lr_at(s, lr, steps)
        for k in p:  # Adam, rồi trung bình trượt của trọng số
            m1[k] = 0.9 * m1[k] + 0.1 * g[k]
            m2[k] = 0.999 * m2[k] + 0.001 * g[k] ** 2
            p[k] -= lr_s * (m1[k] / c1) / (np.sqrt(m2[k] / c2) + 1e-8)
            ema[k] = ema_decay * ema[k] + (1 - ema_decay) * p[k]
    return ema, np.array(losses)


def net_v(p, y=None):
    """Vận tốc của một mạng: v(x, t)."""
    return lambda x, t: forward(p, inputs(x, t, None if y is None else y))


t0 = time.perf_counter()
X_ring = ring_data(20000, seed=10)
p_ring, loss_ring = train_fm(X_ring)
print(f"3.000 bước ({vn(time.perf_counter() - t0, 1)} giây): mất mát 500 bước cuối {vn(loss_ring[-500:].mean(), 2)}, "
      f"mức sàn của bộ khử nhiễu tối ưu {vn(floor, 2)}")  # fmt: skip
assert loss_ring[-500:].mean() - floor < 0.15, "mạng chưa học xong: hãy kiểm tra lại vòng huấn luyện"

Mất mát dừng ngay trên mức sàn 3,59 của mục 4. Ta lấy mẫu từ mạng với cùng các nhiễu của demo:

In [ ]:
Z_EVAL = np.random.default_rng(9).standard_normal((2000, 2))  # 2.000 nhiễu mà bảng reflow của demo dùng
fm_rows = [r for r in REFLOW["rows"] if r["kind"] == "fm" and r["s"] == S_RING]
live_fm = {}
print("số bước │ mạng vừa học │ bộ khử nhiễu tối ưu │ demo: mạng cùng kiểu, 12.000 bước, 3 seed")
for k in (1, 2, 4, 8, 32):
    live_fm[k] = ring_quality(euler(net_v(p_ring), Z_EVAL, grid(k)))[0]
    exact_k = ring_quality(euler(ring_v, Z_EVAL, grid(k)))[0]
    print(f"{k:>7} │ {pct(live_fm[k], 1):>12} │ {pct(exact_k, 1):>19} │ "
          f"{span([r['steps'][str(k)]['inMode'] for r in fm_rows], 1, 100, '%')}")  # fmt: skip
q32 = ring_quality(euler(net_v(p_ring), Z_EVAL, grid(32)))
print(f"32 bước: {q32[1]}/8 cụm có mẫu, bán kính trung bình {vn(q32[2])}")
assert live_fm[1] < 0.01 and live_fm[32] > 0.8 and q32[1] == 8

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2), sharey=True)
for ax, k in zip(axes, (1, 4, 32)):
    xk = euler(net_v(p_ring), Z_EVAL, grid(k))
    ax.scatter(*xk.T, s=3, marker="s", color=ORANGE, alpha=0.5, lw=0)
    ax.scatter(*MEANS.T, s=60, facecolors="none", edgecolors=BLUE, lw=1.5)
    ax.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), aspect="equal", title=f"mạng vừa học, {k} bước")
plt.tight_layout()
vn_axes(fig)
plt.show()

Mạng học trong vài giây đã sinh được cả 8 cụm, và hành xử như bộ khử nhiễu tối ưu: 1 bước rơi về giữa, vài bước đầu chỉ
kéo mẫu về phía vòng tròn. Nhưng nó cần nhiều bước hơn mới vào hẳn cụm (so cột đầu với cột thứ hai ở 8 bước), và mẫu nhòe
hơn quanh mỗi cụm: đó là sai số học. Mạng cùng kiểu của demo, huấn luyện lâu hơn, thu hẹp một phần khoảng cách đó (cột cuối).

## 7 · Đoán nhiễu, đoán ảnh hay đoán vận tốc: cùng một bài toán

DDPM (Ho và cộng sự, 2020) cho mạng đoán nhiễu $\varepsilon$; có mô hình đoán thẳng ảnh sạch $x_0$; flow matching đoán
vận tốc $v = \varepsilon - x_0$. Từ $x_t = (1 - t)x_0 + t\varepsilon$ và bất kỳ một trong ba, ta suy ra hai cái còn lại:

$$\hat x_0 = x_t - t\,v, \qquad \hat\varepsilon = x_t + (1 - t)\,v, \qquad v = \frac{x_t - \hat x_0}{t} = \frac{\hat\varepsilon - x_t}{1 - t}.$$

Vậy **đoán nhiễu cũng là đoán ảnh**: ở mỗi bước, mạng luôn ngầm có một ảnh sạch $\hat x_0$ trong đầu. Ô dưới kiểm tra
các đẳng thức này trên số ngẫu nhiên (đây là đẳng thức thứ nhất mà `reel/diffusion/diff.py` tự kiểm tra).

In [ ]:
rng = np.random.default_rng(0)
x0_r, e_r, t_r = rng.standard_normal((4, 16)), rng.standard_normal((4, 16)), 0.37
xt_r, v_r = (1 - t_r) * x0_r + t_r * e_r, e_r - x0_r
err_conv = max(np.abs(xt_r - t_r * v_r - x0_r).max(), np.abs(xt_r + (1 - t_r) * v_r - e_r).max(),
               np.abs((xt_r - x0_r) / t_r - v_r).max(), np.abs((e_r - xt_r) / (1 - t_r) - v_r).max())  # fmt: skip
print(f"x̂₀ = xₜ − t·v, ε̂ = xₜ + (1 − t)·v và hai cách đổi ngược: sai lệch lớn nhất {sci(err_conv)}")
assert err_conv < 1e-12

Ba cách đoán là cùng một bài hồi quy; đổi qua lại chỉ nhân sai số với một hệ số phụ thuộc $t$, tức là đổi **trọng số**
của mỗi mức nhiễu trong mất mát (Kingma & Gao, 2023). Nhưng hệ số đó có lúc rất lớn: muốn có vận tốc từ một mạng đoán
$\varepsilon$ phải chia cho $1 - t$, từ một mạng đoán $x_0$ phải chia cho $t$. Demo đã huấn luyện cả ba trên cùng mã
MNIST (mạng H = 512, 20.000 bước, 3 seed) và đo sai số vận tốc $\mathbb E\,\lVert \hat v - (\varepsilon - x_0)\rVert^2$
trên các mã kiểm định. Ô dưới chỉ đọc kết quả đó:

In [ ]:
def cells(model, sampler, nfe, key="fd", s=None):
    """Giá trị đã lưu (mỗi seed một số) của một ô trong kết quả quét."""
    rows = [c for c in MN["cells"] if c["model"] == model and c["sampler"] == sampler and c["nfe"] == nfe]
    rows = [c for c in rows if s is None or c.get("s") == s]
    return [c[key] for c in sorted(rows, key=lambda c: c["seed"]) if isinstance(c.get(key), (int, float))]


TS = MN["setup"]["ts"]
TARGET_NAME = {"ae16-fm": "đoán v (flow matching)", "ae16-x0": "đoán x₀", "ae16-eps": "đoán ε"}
param = {m: [r["err"] for r in MN["param"] if r["model"] == m] for m in TARGET_NAME}
print("mạng                     " + "".join(f"t = {vn(t, 2):<6}" for t in TS))
for m, name in TARGET_NAME.items():
    print(f"{name:<25}" + "".join(f"{span([e[i] for e in param[m]], 0):<10}" for i in range(len(TS))))
fd_t = {m: [c["fd"] for c in MN["cells"] if c["model"] == m and c["sampler"] == "euler" and c["nfe"] == 100]
        for m in TARGET_NAME}  # fmt: skip
print("FD sau 100 bước Euler:", " · ".join(f"{TARGET_NAME[m]} {span(v, 1)}" for m, v in fd_t.items()))
print(f"đoán v, t lấy theo logit-normal như SD3: FD {span(cells('ae16-ln', 'euler', 100))} sau 100 bước")

Ở giữa đường, ba mạng sai như nhau (trong dao động giữa các seed). Ở hai đầu thì không: mạng đoán $\varepsilon$ sai gấp
hơn 7 lần ở $t = 0{,}98$ (92–97 so với 12–13), vì gần nhiễu thuần, $x_t \approx \varepsilon$ và một sai số nhỏ trên
$\hat\varepsilon$ bị chia cho $1 - t = 0{,}02$. Mạng đoán $x_0$ có điểm mù ngược lại, ở $t = 0{,}02$. Sai số lớn ở bước
đầu tiên làm hỏng cả quỹ đạo: FD của mạng đoán $\varepsilon$ là 45–75 sau 100 bước. (FD là khoảng cách Fréchet
giữa đặc trưng LeNet của ảnh sinh ra và của 10.000 ảnh test, như Bài 19–20: càng thấp càng tốt.) Đoán $v$ trộn hai
cách và không có điểm mù nào; Stable Diffusion 3 và FLUX đều đoán $v$. Bài báo của SD3 còn lấy $t$ theo phân phối
logit-normal, dồn nhiều mẫu huấn luyện hơn vào giữa đường; trên mã MNIST, cách đó hạ FD sau 100 bước xuống 3,0–3,1.

## 8 · DDPM, DDIM và flow matching: khác bộ giải, không khác mô hình

DDPM viết quá trình thuận khác đi: $t$ là số nguyên từ 1 đến $T = 1.000$, và
$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1 - \bar\alpha_t}\,\varepsilon$ với $\bar\alpha_t = \prod_{i \le t}(1 - \beta_i)$ và
$\beta_t$ tăng tuyến tính từ $10^{-4}$ đến $0{,}02$. Ho và cộng sự (2020) viết rằng họ chọn $T = 1.000$ "để số lần gọi
mạng khi lấy mẫu khớp với các công trình trước", và "không dò tìm" con số đó. DDIM (Song và cộng sự, 2020) lấy mẫu tất
định bằng chính mạng DDPM, trên một dãy con các bước: từ $x_t$, đoán $\hat x_0$ và $\hat\varepsilon$, rồi đặt
$x_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat x_0 + \sqrt{1 - \bar\alpha_{t'}}\,\hat\varepsilon$.

Viết trên đường của flow matching, bước DDIM từ $t$ về $t'$ là $x_{t'} = (1 - t')\,\hat x_0 + t'\,\hat\varepsilon$. Thay
$\hat\varepsilon = (x_t - (1 - t)\hat x_0)/t$ vào, ta được đúng $x_t - (t - t')\,(x_t - \hat x_0)/t$: **một bước Euler**.
Ô dưới kiểm tra điều đó với bộ khử nhiễu chính xác của vòng 8 cụm (đẳng thức thứ hai mà `diff.py` tự kiểm tra), rồi
kiểm tra điều mạnh hơn: chia $x_t$ của DDPM cho $c_t = \sqrt{\bar\alpha_t} + \sqrt{1 - \bar\alpha_t}$ thì được đúng đường
thẳng của flow matching với $\tau = \sqrt{1 - \bar\alpha_t}/c_t$. Vậy cả lần chạy DDIM trên lịch DDPM là Euler trên
một lưới $\tau$ khác.

In [ ]:
BETA = np.linspace(1e-4, 0.02, 1000)
ABAR = np.cumprod(1 - BETA)  # ᾱ_t, t = 1…1.000 ở chỉ số t − 1


def respace(k):
    """Dãy con k chỉ số DDPM cách đều từ T về 1 (như demo)."""
    return np.unique(np.round(np.linspace(1, 1000, k)).astype(int))[::-1] if k > 1 else np.array([1000])


def ddim_ring(z, k):
    """DDIM với bộ khử nhiễu chính xác của vòng, trên lịch DDPM."""
    x, taus = z.copy(), respace(k)
    for i, ti in enumerate(taus):
        ab = ABAR[ti - 1]
        abp = ABAR[taus[i + 1] - 1] if i + 1 < len(taus) else 1.0
        x0 = ring_x0(x, np.sqrt(ab), np.sqrt(1 - ab))
        e = (x - np.sqrt(ab) * x0) / np.sqrt(1 - ab)
        x = np.sqrt(abp) * x0 + np.sqrt(1 - abp) * e
    return x


z = Z_DEMO[0]
t_a, t_b = 0.8, 0.55
x0h = ring_x0(z, 1 - t_a, t_a)
step_euler = z - (t_a - t_b) * (z - x0h) / t_a
step_ddim = (1 - t_b) * x0h + t_b * (z - (1 - t_a) * x0h) / t_a
print(
    f"một bước Euler và một bước DDIM, t = {vn(t_a, 2)} → {vn(t_b, 2)}: lệch {sci(np.abs(step_euler - step_ddim).max())}"
)
assert np.abs(step_euler - step_ddim).max() < 1e-12
for k in (4, 8):
    ab = ABAR[respace(k) - 1]
    c = np.sqrt(ab) + np.sqrt(1 - ab)
    tau = np.r_[np.sqrt(1 - ab) / c, 0.0]
    gap = np.abs(ddim_ring(z, k) - euler(ring_v, z / c[0], tau)).max()
    print(f"DDIM {k} bước = Euler trên lưới τ = {[vn(u, 3) for u in tau]}: lệch {sci(gap)}")
    assert gap < 1e-9
print(f"ᾱ_T = {sci(ABAR[-1])}: ở bước 1.000, x_T vẫn giữ √ᾱ_T = {vn(np.sqrt(ABAR[-1]), 4)} lần ảnh")

Hai bộ giải trùng nhau tới sai số làm tròn. Vậy DDPM, DDIM và flow matching không phải ba loại mô hình: một mạng đoán
$\varepsilon$, $x_0$ hay $v$ chỉ là ba cách viết cùng một bộ khử nhiễu (mục 7), và DDIM chính là Euler. Cái khác nhau là
**chỗ đặt các bước**. Lưới đều của DDPM với 4 bước là $\tau$ = 0,994 → 0,905 → 0,594 → 0,010 → 0: hai bước đầu gần như
phí ở vùng nhiễu thuần, rồi một bước khổng lồ từ 0,594 xuống 0,010. Demo đo đúng điều này trên vòng 8 cụm (ô dưới đọc
bảng của nó, 3 seed nhiễu):

In [ ]:
for k in (4, 8, 32):
    row = {smp: span([r["inMode"] for r in table[f"{smp}/{k}"]], 1, 100, "%") for smp in ("euler", "ddim", "ddpm")}
    print(f"{k:>2} bước, mẫu trong một cụm: Euler (lưới t đều) {row['euler']:>11} · DDIM (lưới DDPM) {row['ddim']:>11} · "
          f"DDPM ngẫu nhiên {row['ddpm']}")  # fmt: skip

Cùng bộ khử nhiễu hoàn hảo, cùng 4 bước: lưới đều theo $t$ cho 76,8–78,4% mẫu trong một cụm, lưới DDPM chỉ 1,8–1,9%.
Với 32 bước thì mọi cách đều tốt. Bộ giải ngẫu nhiên của DDPM (mỗi bước thêm nhiễu mới) còn kém DDIM một chút khi ít bước, và chỉ
bằng hoặc hơn khi nhiều bước.

Trên MNIST cũng vậy. Demo huấn luyện cả mạng DDPM ($T = 1.000$, đoán $\varepsilon$) và mạng flow matching trên cùng các
mã, 3 seed mỗi mạng, rồi đo FD trên 10.000 mẫu cho mỗi bộ giải và số lần gọi mạng:

In [ ]:
LINES = [("ae16-fm", "euler", "flow matching, Euler"), ("ae16-ddpm", "ddim", "DDPM, bộ giải DDIM"),
         ("ae16-ddpm", "ddpm", "DDPM, ngẫu nhiên")]  # fmt: skip
print("lần gọi mạng │ " + " │ ".join(f"{name:<20}" for _, _, name in LINES))
for k in (1, 2, 4, 8, 16, 32, 100, 1000):
    print(
        f"{vn(k, 0):>11} │ "
        + " │ ".join(f"{span(cells(m, smp, k)) if cells(m, smp, k) else '—':<20}" for m, smp, _ in LINES)
    )
print(
    f"1.000 bước Euler (seed 0): FD {vn(cells('ae16-fm', 'euler', 1000)[0])} · 1/√ᾱ_T = {vn(1 / np.sqrt(ABAR[-1]), 0)}"
)
for m, smp, name in LINES[::2]:
    print(f"1 lần gọi, {name}: độ lệch chuẩn của mã {span(cells(m, smp, 1, 'codeStd'))}, "
          f"precision {span(cells(m, smp, 1, 'P'), 2)}, recall {span(cells(m, smp, 1, 'R'), 2)}")  # fmt: skip

Ba điều cần đọc trong bảng (FD thấp là tốt):
- Ở 8 lần gọi mạng, cùng một mạng DDPM cho FD 16,1–18,2 với bộ giải ngẫu nhiên và 10,4–12,1 với DDIM: ít bước thì thêm
  nhiễu mới là thiệt.
- Flow matching với Euler ở 8 lần gọi đạt 6,1–6,6, tốt hơn DDIM trên mạng DDPM. Một phần lợi thế đó là lưới $t$ đều,
  không phải mục tiêu huấn luyện (ô trên).
- 32 bước Euler cho 3,7–3,9; 1.000 bước (seed 0) cho 3,45. Con số 1.000 là của lúc huấn luyện DDPM, không phải số bước
  cần khi sinh ảnh.

Một bước của mạng DDPM (FD 58–70) trông "tốt hơn" một bước của flow matching (247–284), nhưng không phải vì nó đúng hơn. Ở
$t = T$, ảnh chỉ còn chiếm $\sqrt{\bar\alpha_T} = 0{,}0064$ của $x_T$ (ô trên), nên $\hat x_0 = (x_T - \sqrt{1 - \bar\alpha_T}\,
\hat\varepsilon)/\sqrt{\bar\alpha_T}$ nhân sai số của $\hat\varepsilon$ lên 157 lần, đúng điểm mù của mục 7. Mã sinh ra có độ
lệch chuẩn 5,1–5,2, gấp 5 lần mã thật: mẫu đa dạng (recall 0,65–0,73) nhưng phần lớn không giống chữ số nào (precision
0,33–0,41). Một bước flow matching thì ngược lại: mọi mẫu là cùng một "chữ số trung bình" (precision 0,84–0,99, recall
gần 0). FD gộp cả hai thứ thành một con số; cả hai đều hỏng.

## 9 · Làm thẳng đường đi: reflow

Một bước Euler hỏng vì trường vận tốc cong (mục 5): nó là trung bình của mọi đoạn thẳng nối một nhiễu **ngẫu nhiên**
với một điểm dữ liệu **ngẫu nhiên**. Reflow (Liu và cộng sự, 2022) đổi cách ghép cặp: lấy mỗi nhiễu $z$, chạy mô hình đã
có để biết nó đưa $z$ tới điểm $x$ nào, rồi học lại flow matching trên các cặp $(x, z)$ cố định đó. Các cặp này được
ghép theo đường đi của một phương trình vi phân, mà hai đường đi như thế không bao giờ cắt nhau; nên các đoạn thẳng nối
chúng ít cắt nhau hơn hẳn, mỗi $x_t$ gần như chỉ thuộc về một cặp, và trường vận tốc mới gần như thẳng.

Ô dưới làm đúng như vậy trên vòng 8 cụm: ghép 10.000 nhiễu với điểm cuối của 100 bước Euler trên trường chính xác, rồi
huấn luyện cùng mạng, cùng công thức như mục 6, chỉ khác là `E` cố định.

In [ ]:
t0 = time.perf_counter()
Z_pair = np.random.default_rng(10).standard_normal((10000, 2))
X_pair = euler(ring_v, Z_pair, grid(100))  # điểm mà trường chính xác đưa mỗi nhiễu tới
p_rf, _ = train_fm(X_pair, E=Z_pair)
print(f"reflow: {vn(time.perf_counter() - t0, 1)} giây")


def straightness(v, z, steps=200):
    """Trung vị của (độ dài đường đi) / (dây cung) qua các điểm z."""
    path = [z]
    x = euler(v, z, grid(steps), path)
    length = np.hypot(*np.diff(np.array(path), axis=0).transpose(2, 0, 1)).sum(0)
    return np.median(length / np.hypot(*(x - z).T))


live_rf = {k: ring_quality(euler(net_v(p_rf), Z_EVAL, grid(k)))[0] for k in (1, 2, 4, 8, 32)}
print("mẫu trong một cụm   │ " + " · ".join(f"{k} bước" for k in live_rf))
print("flow matching (mục 6)│ " + " · ".join(pct(live_fm[k], 0) for k in live_rf))
print("reflow              │ " + " · ".join(pct(live_rf[k], 0) for k in live_rf))
print(f"đường đi / dây cung: flow matching {vn(straightness(net_v(p_ring), Z_EVAL[:300]), 2)}, "
      f"reflow {vn(straightness(net_v(p_rf), Z_EVAL[:300]), 3)}")  # fmt: skip
assert live_rf[1] > 0.5 > live_fm[1]

Một bước của mạng reflow đã đưa khoảng ba phần tư số mẫu vào đúng cụm, so với 0% của mạng flow matching thường, và
đường đi của nó gần như thẳng tắp (tỉ lệ ≈ 1). Demo dùng một mạng reflow huấn luyện lâu hơn, trên nhiều cặp hơn; tệp
`diffusion-reflow.json` có trọng số float32 của nó, nên ta chạy được đúng mạng đó và so với bảng của demo:

In [ ]:
p_rf_demo = {k: unpack(t) for k, t in REFLOW["net"].items()}
setup = REFLOW["setup"]
print(f"mạng reflow của demo: H = {setup['H']}, {vn(setup['pairs'], 0)} cặp ({setup['coupling']}), "
      f"{vn(setup['steps'], 0)} bước huấn luyện")  # fmt: skip
row0 = next(r for r in REFLOW["rows"] if r["kind"] == "rf2" and r["s"] == S_RING and r["seed"] == 0)
rf_rows = [r for r in REFLOW["rows"] if r["kind"] == "rf2" and r["s"] == S_RING]
for k in (1, 2, 4, 8, 16, 32):
    q = ring_quality(euler(net_v(p_rf_demo), Z_EVAL, grid(k)))[0]
    print(f"{k:>2} bước: {pct(q, 2)} (demo, seed 0: {pct(row0['steps'][str(k)]['inMode'], 2)}; 3 seed: "
          f"{span([r['steps'][str(k)]['inMode'] for r in rf_rows], 1, 100, '%')})")  # fmt: skip
    assert abs(q - row0["steps"][str(k)]["inMode"]) < 0.002, "khác với demo: hãy kiểm tra lại mạng reflow"

Trên MNIST, demo làm reflow cho mạng flow matching của mã 16 số (mỗi mã huấn luyện một cặp, từ 100 bước Euler,
3 seed):

In [ ]:
for k in (1, 2, 4, 8, 32):
    print(
        f"{k:>2} bước: FD flow matching {span(cells('ae16-fm', 'euler', k)):>11} · reflow {span(cells('ae16-rf2', 'euler', k))}"
    )
print(f"1 bước flow matching: LeNet đọc {span(cells('ae16-fm', 'euler', 1, 'histMax'), 0, 100, '%')} số mẫu là cùng một "
      f"chữ số; reflow: {span(cells('ae16-rf2', 'euler', 1, 'histMax'), 0, 100, '%')} (dữ liệu thật: mỗi chữ số ~10%)")  # fmt: skip
print(f"GAN Bài 20: FD {span(MN['ref']['gan'])}")

Một bước của flow matching thường cho FD 247–284: mọi mã co về gần mã trung bình (mục 10 đo điều này), và LeNet đọc
46–94% số mẫu là cùng một chữ số. Sau reflow, một bước cho 5,4–5,6, tốt hơn GAN của Bài 20 (7,9–8,0). Cái
giá: ở 32 bước, mạng reflow (4,0–4,2) kém mạng gốc (3,7–3,9), một phần vì nó học từ chính mẫu của mạng gốc, kể cả sai
số của mạng đó. Các mô hình sinh ảnh trong 1–4 bước (SDXL Turbo, FLUX.1 [schnell], LCM) dùng những cách **chưng cất** mạnh hơn cùng ý
tưởng này: dạy một mạng học trò đi một bước tới chỗ mà mạng thầy đi nhiều bước.

## 10 · Khuếch tán trong mã của autoencoder: chữ số từ nhiễu

Giờ đến MNIST. Mô hình khuếch tán không chạm vào điểm ảnh: nó sinh mã 16 số, rồi bộ giải mã của Bài 19 vẽ mã thành ảnh.
Ta dùng đúng mạng flow matching mà demo chạy trong trình duyệt (H = 256, 40.000 bước, seed 0) và cùng các nhiễu mà
`sweep.py` đã dùng để đánh giá nó (`default_rng(7)`, float32): cả 10.000 cho 1 và 4 bước, 2.000 đầu tiên cho các lần
chạy dài hơn. Kết quả quét có lưu độ lệch chuẩn của 10.000 mã sinh ra cho từng số bước; ta tính lại con số đó để kiểm
tra mạng của mình.

Heun là bộ giải bậc hai: đi thử một bước Euler, đo vận tốc ở điểm đến, rồi đi lại bằng trung bình hai vận tốc. Mỗi bước
tốn hai lần gọi mạng, trừ bước cuối (đáp xuống $t = 0$ bằng Euler), nên $k$ bước Heun tốn $2k - 1$ lần gọi.

In [ ]:
def heun(v, z, ts):
    x = z.copy()
    for a, b in itertools.pairwise(ts):
        va = v(x, a)
        if b > 0:
            va = 0.5 * (va + v(x - (a - b) * va, b))
        x = x - (a - b) * va
    return x


NET_FM = {k: unpack(t) for k, t in NETS["nets"]["fmB"]["p"].items()}
v_fm = net_v(NET_FM)
Z_NOISE = np.random.default_rng(7).standard_normal((10000, 16)).astype(np.float32).astype(np.float64)
print(f"mạng của trình duyệt: H = {NETS['nets']['fmB']['H']}, {vn(NETS['nets']['fmB']['steps'], 0)} bước huấn luyện; "
      f"mạng của kết quả quét: H = {MN['setup']['H']}, {vn(MN['setup']['steps'], 0)} bước, 3 seed")  # fmt: skip
t0 = time.perf_counter()
traj = []  # 32 bước Euler trên 2.000 nhiễu đầu tiên; giữ lại mọi xₜ cho mục dưới
codes = {
    "1 bước Euler": (euler(v_fm, Z_NOISE, grid(1)), "euler", 1),
    "4 bước Euler": (euler(v_fm, Z_NOISE, grid(4)), "euler", 4),
    "32 bước Euler": (euler(v_fm, Z_NOISE[:2000], grid(32), traj), "euler", 32),
    "5 bước Heun": (heun(v_fm, Z_NOISE[:2000], grid(5)), "heun", 9),
}
print(f"lấy mẫu: {vn(time.perf_counter() - t0, 1)} giây")
for name, (Zs, smp, nfe) in codes.items():
    stored = cells("ae16-fmB", smp, nfe, "codeStd")[0]
    print(f"{name:>13} ({nfe:>2} lần gọi mạng, {vn(len(Zs), 0)} mẫu): độ lệch chuẩn của mã {vn(Zs.std(), 3)} "
          f"(demo, 10.000 mẫu: {vn(stored, 3)}), FD đã lưu {vn(cells('ae16-fmB', smp, nfe)[0])}")  # fmt: skip
    assert abs(Zs.std() - stored) < (0.002 if len(Zs) == 10000 else 0.01), "khác với demo: hãy kiểm tra lại mạng"

Mạng của ta cho đúng các con số mà demo lưu, nên đây đúng là mạng của demo. Hình dưới giải mã 16 mã đầu tiên của mỗi lần
chạy. Ô đầu tiên giải mã chính nhiễu $\mathcal N(0, I)$ (0 bước): đó là cách lấy mẫu của Bài 19, với cùng bộ giải mã.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2.6))
show_tile(axes[0], decode(Z_NOISE[:16]), 4, 4, "giải mã nhiễu N(0, I)")
for ax, (name, (Zs, _, nfe)) in zip(axes[1:], codes.items()):
    show_tile(ax, decode(Zs[:16]), 4, 4, f"{name}\n({nfe} lần gọi mạng)")
plt.tight_layout()
plt.show()


def spread(Z):
    """Khoảng cách trung bình từ mỗi mã tới trung bình của các mã: mẫu đa dạng đến đâu."""
    return np.sqrt(((Z - Z.mean(0)) ** 2).sum(1)).mean()


print(f"độ trải của mã: mã thật {vn(spread(Zc))} · " + " · ".join(f"{n} {vn(spread(c[0]))}" for n, c in codes.items()))
print(f"độ lệch chuẩn quanh trung bình của chính các mẫu: mã thật {vn(np.sqrt(((Zc - Zc.mean(0)) ** 2).mean()))}, "
      f"1 bước Euler {vn(np.sqrt(((codes['1 bước Euler'][0] - codes['1 bước Euler'][0].mean(0)) ** 2).mean()))}")  # fmt: skip
z1 = codes["1 bước Euler"][0]
print(f"1 bước: trung bình của mẫu cách mã trung bình thật {vn(np.linalg.norm(z1.mean(0) - Zc.mean(0)))}; "
      f"độ lệch chuẩn của mọi số trong mã vẫn là {vn(z1.std())}, vì chính mã trung bình có 16 số với độ lệch chuẩn "
      f"{vn(Zc.mean(0).std())}")  # fmt: skip
assert spread(z1) < 0.2 * spread(Zc)

Giải mã thẳng nhiễu cho những nét nguệch ngoạc: bộ giải mã của một autoencoder không có KL chỉ biết vẽ những mã nằm
gần mã thật, còn $\mathcal N(0, I)$ phủ cả vùng trống giữa chúng (Bài 19). Một bước Euler cho 16 ảnh gần như y hệt nhau:
độ trải của mã chỉ còn 0,28, so với 3,38 của mã thật (độ lệch chuẩn quanh trung bình của chính các mẫu: 0,08 so với
0,86). Thống kê "độ lệch chuẩn của mọi số trong mã" mà demo in ra vẫn là 0,52 ở đây, chỉ vì 16 số của chính mã trung
bình đã có độ lệch chuẩn đó; nó không đo độ đa dạng. 4 bước đã
cho chữ số đọc được nhưng còn mờ, 32 bước Euler hay 5 bước Heun (9 lần gọi mạng) thì sắc nét.

### Nét lớn hiện trước, chi tiết đến sau
Ở mỗi bước, mạng có một ảnh đoán $\hat x_0 = x_t - t\,v$ (mục 7). Hình dưới theo 4 mẫu qua 32 bước Euler: hàng trên của
mỗi mẫu giải mã $x_t$, hàng dưới giải mã $\hat x_0$, ở 9 thời điểm $t$ ghi trên cột.

In [ ]:
xt_all = [Z_NOISE[:2000], *traj]  # x_t ở t = 1, 31/32, …, 0
ts32 = grid(32)
# bước Euler x_{t−Δt} = x_t − Δt·v, nên v = (x_t − x_{t−Δt})/Δt và x̂₀ = x_t − t·v: không cần gọi lại mạng
x0_hats = [xt_all[i] - ts32[i] * (xt_all[i] - xt_all[i + 1]) / (ts32[i] - ts32[i + 1]) for i in range(32)] + [traj[-1]]
pick = np.round(np.linspace(0, 32, 9)).astype(int)
fig, axes = plt.subplots(8, 9, figsize=(9, 8.6))
for r in range(4):
    for c, i in enumerate(pick):
        for row, src in ((2 * r, xt_all), (2 * r + 1, x0_hats)):
            axes[row, c].imshow(decode(src[i][r]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
            axes[row, c].set_xticks([])
            axes[row, c].set_yticks([])
        axes[0, c].set_title(f"t = {vn(ts32[i], 2)}", fontsize=9)
    axes[2 * r, 0].set_ylabel("xₜ", fontsize=9)
    axes[2 * r + 1, 0].set_ylabel("x̂₀", fontsize=9)
plt.tight_layout()
plt.show()

$x_t$ giải mã ra một đám mờ cho tới khá muộn, nhưng $\hat x_0$ đã có dáng chữ số từ sớm: ở $t = 0{,}75$ nó là một bóng mờ
của chữ số trung bình, đến $t = 0{,}5$ đã đọc được, và nửa sau của quỹ đạo chỉ sửa nét. Ta đo điều đó bằng một bộ phân loại
đơn giản trên mã: chữ số đa số trong 5 mã huấn luyện gần nhất (nó đoán đúng 90,8% số mã thật khi bỏ chính mã đó ra).
Ở mỗi bước, tỉ lệ trong 2.000 mẫu mà $\hat x_0$ đã mang đúng chữ số của ảnh cuối cùng:

In [ ]:
def knn_digit(Z, k=5, exclude_self=False):
    """Chữ số đa số trong k mã huấn luyện gần nhất (hòa thì lấy chữ số nhỏ hơn)."""
    d2 = (Z**2).sum(1)[:, None] - 2 * Z @ Zc.T + (Zc**2).sum(1)[None]
    if exclude_self:
        np.fill_diagonal(d2, np.inf)
    nn = np.argpartition(d2, k - 1, axis=1)[:, :k]
    return (y_img[nn][:, :, None] == np.arange(10)).sum(1).argmax(1)


judge_acc = (knn_digit(Zc, exclude_self=True) == y_img).mean()
final = knn_digit(x0_hats[-1])
same = {i: (knn_digit(x0_hats[i]) == final).mean() for i in pick}
demo_same = {round(r["t"], 4): r["same"] for r in MN["traj"]}
print(f"bộ phân loại 5 láng giềng trên mã: đúng {pct(judge_acc, 1)} mã huấn luyện (bỏ chính nó ra)")
for i in pick:
    t = ts32[i]
    print(f"t = {vn(t, 2)}: x̂₀ đã mang đúng chữ số cuối {pct(same[i], 0):>4} "
          f"(demo, LeNet trên ảnh: {pct(demo_same[round(t, 4)], 0)})")  # fmt: skip
assert same[16] > 0.75 and same[0] < 0.3

Hai bộ phân loại khác nhau (của ta đọc mã, của demo là LeNet đọc ảnh) cho cùng một câu chuyện: lúc $t = 1$ chỉ khoảng 10%
(đoán bừa một trong 10 chữ số), đến $t = 0{,}5$ đã 86%. Mô hình khuếch tán quyết định **chữ số nào** trong nửa đầu,
rồi dùng nửa sau để vẽ chi tiết. Với ảnh thật, Dieleman (2024) gọi đây là "tự hồi quy theo tần số": nhiễu Gauss xóa các
chi tiết nhỏ (tần số cao) trước, nên khử nhiễu dựng lại hình dáng lớn trước.

### Tốt đến đâu? Đọc kết quả quét
FD đo khoảng cách giữa 10.000 mẫu và 10.000 ảnh test, qua đặc trưng của LeNet (Bài 19–20; thấp là tốt). Recall đo phần
ảnh thật mà mẫu "phủ" được (Bài 20). Mọi dòng dùng cùng một bộ giải mã; chỉ khác cách lấy mã:

In [ ]:
def base(space, kind, key="fd"):
    return [b[key] for b in sorted(MN["base"], key=lambda b: b["seed"]) if b["space"] == space and b["kind"] == kind]


floors = {sp: [f["fd"] for f in MN["floors"] if f["space"] == sp] for sp in ("ae16", "vae16")}
rows = [
    ("giải mã chính mã của ảnh test (trần)", floors["ae16"], None),
    ("flow matching, 5 bước Heun (9 lần gọi)", cells("ae16-fm", "heun", 9), cells("ae16-fm", "heun", 9, "R")),
    ("flow matching, 100 bước Euler", cells("ae16-fm", "euler", 100), cells("ae16-fm", "euler", 100, "R")),
    ("flow matching, 32 bước Euler", cells("ae16-fm", "euler", 32), cells("ae16-fm", "euler", 32, "R")),
    ("hỗn hợp 10 Gauss khớp trên mã (Bài 19)", base("ae16", "gmm"), base("ae16", "gmm", "R")),
    ("GAN của Bài 20 (sinh thẳng điểm ảnh)", MN["ref"]["gan"], None),
    ("một Gauss khớp trên mã", base("ae16", "gauss"), base("ae16", "gauss", "R")),
    ("N(0, I), như mục giải mã nhiễu ở trên", base("ae16", "prior"), base("ae16", "prior", "R")),
]
print(f"{'cách lấy mã (autoencoder 16 chiều, 3 seed)':<44} {'FD':>11}  recall")
for name, fd, rc in rows:
    print(f"{name:<44} {span(fd):>11}  {span(rc, 2) if rc else '—'}")
print(f"\nmã của VAE (β = 1) thay cho autoencoder: flow matching 100 bước {span(cells('vae16-fm', 'euler', 100))}, "
      f"N(0, I) {span(base('vae16', 'prior'))}, trần {span(floors['vae16'])}")  # fmt: skip
print(f"hai tập ảnh thật: recall {span(MN['ref']['realR'], 2)}")

Cùng một bộ giải mã của Bài 19, chỉ khác chỗ lấy mã:
- Lấy mã từ $\mathcal N(0, I)$ cho FD 75–94. Một mô hình flow matching nhỏ sinh mã cho 3,4–3,6 sau 100 bước Euler, tốt
  hơn hẳn GAN của Bài 20 (7,9–8,0) và hỗn hợp Gauss của Bài 19 (7,0–7,6). Trần của bộ giải mã (giải mã chính mã của ảnh
  test) là 0,8–0,9, nên phần lớn sai số còn lại nằm ở mô hình của mã, không ở bộ giải mã.
- 5 bước Heun (9 lần gọi mạng) còn cho FD thấp hơn 100 bước Euler, nhưng recall thấp hơn một chút (0,76 so với
  0,78–0,79). FD gộp độ giống thật và độ phủ thành một con số, nên hai cách lấy mẫu có FD gần nhau vẫn có thể khác nhau
  ở độ phủ.
- Mã của VAE gần $\mathcal N(0, I)$ hơn hẳn (FD 31,2–31,8 khi giải mã nhiễu, so với 75–94), nhưng mô hình khuếch tán
  trên mã VAE lại kém hơn (5,3–6,0), vì bộ giải mã của VAE vẽ kém hơn (trần 2,0–2,1). Khi đã có mô hình khuếch tán lo phần phân
  phối, autoencoder chỉ cần nén tốt. Stable Diffusion cũng chọn KL rất nhỏ ($10^{-6}$, Bài 19) vì lý do đó.

## 11 · Bộ khử nhiễu tối ưu chỉ biết chép

Ở mục 4, bộ khử nhiễu tối ưu của vòng 8 cụm dùng **phân phối thật** của dữ liệu. Trong thực tế ta chỉ có $n$ ảnh huấn
luyện, và cực tiểu thật sự của mất mát là bộ khử nhiễu tối ưu cho phân phối đều trên đúng $n$ điểm đó:

$$\mathbb E[x_0 \mid x_t] = \sum_{i=1}^{n} w_i\, z_i, \qquad w = \mathrm{softmax}\Big(-\frac{\lVert x_t - (1 - t)\,z_i \rVert^2}{2t^2}\Big),$$

trung bình có trọng số softmax của các mã huấn luyện $z_i$. Khi $t$ nhỏ, softmax dồn hết trọng số cho mã gần nhất.

In [ ]:
def exact_x0(xt, t, Z):
    """E[x₀ | xₜ] khi x₀ đều trên các hàng của Z (xₜ = (1 − t)x₀ + tε)."""
    lw = -((xt**2).sum(1)[:, None] - 2 * (1 - t) * xt @ Z.T + (1 - t) ** 2 * (Z**2).sum(1)[None]) / (2 * t * t)
    w = np.exp(lw - lw.max(1, keepdims=True))
    return (w / w.sum(1, keepdims=True)) @ Z


def exact_sample(Z, z, steps=50):
    """Euler trên vận tốc (xₜ − x̂₀)/t của bộ khử nhiễu tối ưu; bước cuối đáp thẳng xuống x̂₀ (như demo)."""
    ts, x = grid(steps), z.copy()
    for a, b in itertools.pairwise(ts[:-1]):
        x = x - (a - b) * (x - exact_x0(x, a, Z)) / a
    return exact_x0(x, ts[-2], Z)

### 🤔 Dự đoán trước
Lấy 1.000 mẫu bằng bộ khử nhiễu tối ưu cho 1.000 mã huấn luyện. Bao nhiêu mẫu là **bản sao** của một mã huấn luyện, và
chúng trúng bao nhiêu mã khác nhau?

In [ ]:
memo = {e["n"]: e for e in MN["memo"]["exact"]}
S_exact = {}
for n in (100, 1000):
    S_exact[n] = exact_sample(Zc[:n], Z_NOISE[:1000])
    d, idx = nearest(S_exact[n], Zc[:n])
    expect = n * (1 - (1 - 1 / n) ** 1000)  # số mã khác nhau khi rút 1.000 lần đều, có hoàn lại
    print(f"{vn(n, 0):>5} mã huấn luyện: khoảng cách lớn nhất tới mã gần nhất {vn(d.max(), 9)}, "
          f"{vn(len(np.unique(idx)), 0)} mã khác nhau (demo: {vn(memo[n]['distinct'], 0)}; rút đều: {vn(expect, 1)})")  # fmt: skip
    assert d.max() < 1e-6 and abs(len(np.unique(idx)) - memo[n]["distinct"]) <= 5, "khác với demo"
print("demo, mọi cỡ tập:", " · ".join(f"{vn(e['n'], 0)} mã → {pct(e['copy'], 0)} bản sao" for e in MN["memo"]["exact"]))

Mọi mẫu đều trùng khít một mã huấn luyện (khoảng cách đúng bằng 0), ở mọi cỡ tập mà demo thử, kể cả 55.000 mã. Với 1.000
mã, 1.000 mẫu trúng 637 mã khác nhau, gần đúng số mã khác nhau khi rút ngẫu nhiên đều 1.000 lần (632): bộ khử nhiễu tối
ưu chỉ là một cách **chọn ngẫu nhiên một ảnh huấn luyện**. Cực tiểu thật sự của mất mát khuếch tán là học thuộc lòng.

Vậy vì sao mạng thật lại sinh được chữ số mới? Vì nó **không** đạt được cực tiểu đó: một mạng nhỏ, trơn, huấn luyện có
hạn, nội suy giữa các mã thay vì nhớ từng mã (thiên kiến quy nạp). Khi ít dữ liệu, mạng đủ sức nhớ hết. Ô dưới huấn luyện
cùng mạng của mục 6 (bây giờ trên mã 16 số) với 10 và với 500 mã, rồi đếm mẫu "là bản sao": khoảng cách tới mã huấn
luyện gần nhất nhỏ hơn mức mà 99% mã chưa thấy (mã của 500 ảnh còn lại) đều vượt qua. Đó là cách đo của demo và Bài 20.

In [ ]:
held = Zc[500:]  # 500 mã chưa từng đưa vào huấn luyện
z_copy = Z_NOISE[:500]
live_copy = {}
for n in (10, 500):
    t0 = time.perf_counter()
    p_n, _ = train_fm(Zc[:n], steps=1500, lr=3e-3)
    S_n = euler(net_v(p_n), z_copy, grid(32))
    thr = np.quantile(nearest(held, Zc[:n])[0], 0.01)  # 99% mã chưa thấy cách xa hơn mức này
    live_copy[n] = (nearest(S_n, Zc[:n])[0] < thr).mean()
    print(f"học trên {vn(n, 0):>3} mã ({vn(time.perf_counter() - t0, 1)} giây): {pct(live_copy[n], 1)} mẫu là bản sao")
assert live_copy[10] > 0.4 and live_copy[500] < 0.05

Cùng một mạng, cùng số bước huấn luyện: học trên 10 mã thì phần lớn mẫu là bản sao, trên 500 mã thì gần như không. Demo
đo điều này với mạng lớn (H = 512, 20.000 bước, 3 seed) trên 100 đến 55.000 mã, và đo cả trong không gian điểm ảnh (mẫu
đã giải mã so với chính các ảnh MNIST huấn luyện):

In [ ]:
print(f"{'số mã huấn luyện':>16} │ bộ khử nhiễu tối ưu │ mạng: bản sao (mã) │ (điểm ảnh) │ FD        │ recall")
for n in (100, 1000, 10000, 55000):
    model = "ae16-fm" if n == 55000 else f"ae16-n{n}"
    c = {k: cells(model, "euler", 100, k) for k in ("copy", "copyPix", "fd", "R")}
    print(f"{vn(n, 0):>16} │ {pct(memo[n]['copy'], 0):>19} │ {span(c['copy'], 1, 100, '%'):>18} │ "
          f"{span(c['copyPix'], 1, 100, '%'):>10} │ {span(c['fd']):<9} │ {span(c['R'], 2)}")  # fmt: skip
print(f"GAN của Bài 20 trên 1.000 ảnh: {span(MN['ref']['ganCopyN1000'], 1, 100, '%')} mẫu là bản sao (điểm ảnh)")

Học trên 100 mã, 96,2–97,9% mẫu là bản sao của một mã huấn luyện, và recall chỉ 0,12–0,13: mô hình chép lại tập huấn luyện
nhỏ của nó. Trên 55.000 mã, chỉ còn 0,1–0,2%. Ở 1.000 ảnh, 4,3–4,6% mẫu gần như chép một ảnh huấn luyện tính theo điểm
ảnh, so với 0,0–0,1% của GAN Bài 20 cùng cỡ dữ liệu. Ở quy mô lớn cũng vậy: Carlini và cộng sự (2023) sinh 500 ảnh cho
mỗi chú thích của 350.000 ảnh bị lặp nhiều nhất trong dữ liệu của Stable Diffusion (175 triệu lần sinh) và trích được 94
ảnh huấn luyện gần như y hệt; trong thí nghiệm trên CIFAR-10 của họ, mô hình khuếch tán rò rỉ ảnh huấn luyện nhiều hơn
GAN hơn hai lần. Kadkhodaie và
cộng sự (2023) cho thấy chiều ngược lại: khi đủ dữ liệu, hai mạng học trên hai nửa rời nhau của một tập ảnh sinh ra gần
như cùng một ảnh từ cùng một nhiễu, tức là chúng học cùng một phân phối chứ không nhớ ảnh.

## 12 · Hướng dẫn không cần bộ phân loại: bám nhãn hay đa dạng

Muốn sinh một chữ số được chọn, ta cho mạng thêm nhãn: $v_\theta(x_t, t, c)$. Demo huấn luyện **một** mạng như thế, thay
nhãn bằng "không nhãn" ∅ cho 10% số mẫu trong mỗi lô, nên cùng mạng đó vừa là mạng có điều kiện vừa là mạng không điều
kiện. Khi sinh, hướng dẫn không cần bộ phân loại (Ho & Salimans, 2022) ngoại suy từ dự đoán không nhãn về phía dự đoán
có nhãn:

$$\tilde v = v_\varnothing + s\,(v_c - v_\varnothing).$$

$s = 0$ là chỉ dùng nhánh không nhãn, $s = 1$ là mạng có điều kiện thường, $s > 1$ là "hướng dẫn": đẩy xa hơn theo hướng
làm ảnh giống chữ số $c$ hơn. Mỗi bước tốn hai lần gọi mạng. Ô dưới chạy mạng có điều kiện mà demo dùng trong trình duyệt
(32 bước Euler, 2.000 mẫu, 200 mẫu cho mỗi chữ số) và kiểm tra độ lệch chuẩn của mã với kết quả quét. Nó đo độ đúng bằng
bộ phân loại 5 láng giềng của mục 10, và precision/recall (Bài 20) của 1.000 mẫu so với 1.000 mã thật.

### 🤔 Dự đoán trước
Đi từ $s = 1$ lên $s = 3$, độ đúng chữ số tăng. Còn recall, phần các kiểu viết thật mà mẫu phủ được, sẽ tăng, giữ nguyên
hay giảm?

In [ ]:
NET_CFG = {k: unpack(t) for k, t in NETS["nets"]["cfgB"]["p"].items()}


def v_cfg(y, s):
    """Vận tốc có hướng dẫn với nhãn y (mảng 0–9) và hệ số s."""
    null = np.full(len(y), 10)

    def v(x, t):
        if s == 0:
            return forward(NET_CFG, inputs(x, t, null))
        vc = forward(NET_CFG, inputs(x, t, y))
        if s == 1:
            return vc
        vu = forward(NET_CFG, inputs(x, t, null))
        return vu + s * (vc - vu)

    return v


def prec_rec(real, fake, k=3):
    """Precision (mẫu nằm trong quả cầu k-NN của một điểm thật) và recall (điểm thật nằm trong quả cầu của một mẫu)."""

    def radius(A):
        d2 = np.maximum((A**2).sum(1)[:, None] - 2 * A @ A.T + (A**2).sum(1)[None], 0)
        np.fill_diagonal(d2, np.inf)
        return np.sqrt(np.partition(d2, k - 1, axis=1)[:, k - 1])

    D = np.sqrt(np.maximum((fake**2).sum(1)[:, None] - 2 * fake @ real.T + (real**2).sum(1)[None], 0))
    return (D <= radius(real)[None]).any(1).mean(), (D.T <= radius(fake)[None]).any(1).mean()


y_req = np.arange(2000) % 10  # chữ số yêu cầu, như kết quả quét
guided = {}
for s_g in (0, 1, 3, 9):
    Zg = euler(v_cfg(y_req, s_g), Z_NOISE[:2000], grid(32))
    acc = (knn_digit(Zg) == y_req).mean()
    P, Rc = prec_rec(Zc, Zg[:1000])
    guided[s_g] = {"Z": Zg, "acc": acc, "P": P, "R": Rc}
    stored = cells("ae16-cfgB", "cfg", 32, "codeStd", s=s_g)[0]
    print(f"s = {s_g}: đúng chữ số yêu cầu {pct(acc, 1):>6} (5 láng giềng trên mã) · precision {vn(P)} · recall {vn(Rc)}"
          f" · độ lệch chuẩn của mã {vn(Zg.std(), 3)} (demo: {vn(stored, 3)})")  # fmt: skip
    assert abs(Zg.std() - stored) < 0.01, "khác với demo: hãy kiểm tra lại mạng có điều kiện"
P_real, R_real = prec_rec(Zc[:500], Zc[500:])
print(f"hai nửa của 1.000 mã thật: precision {vn(P_real)}, recall {vn(R_real)}")
assert guided[3]["acc"] > guided[1]["acc"] and guided[3]["R"] < guided[1]["R"] - 0.15

Mạng của ta cho đúng độ lệch chuẩn mà demo lưu cho mỗi $s$ (chỉ lệch ở chữ số thứ ba, vì ta lấy 2.000 thay vì 10.000
mẫu). Bộ phân loại 5 láng giềng của mục 10 thấy độ đúng tăng từ 90,6% lên 99,7% khi $s$ đi từ 1 lên 3, trong khi recall
trên mã giảm từ 0,86 xuống 0,55. Hình dưới vẽ 16 mẫu "chữ số 3" ở mỗi $s$, cùng các nhiễu:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, (s_g, g) in zip(axes, guided.items()):
    sel = np.flatnonzero(y_req == 3)[:16]
    title = "s = 0 (bỏ nhãn)" if s_g == 0 else f"s = {s_g}"
    show_tile(ax, decode(g["Z"][sel]), 4, 4, f"{title}: đúng {pct(g['acc'], 0)}")
plt.tight_layout()
plt.show()

Bỏ nhãn ($s = 0$) cho đủ loại chữ số. $s = 1$ đã cho phần lớn là số 3, với đủ kiểu viết. $s = 3$ cho số 3 nào cũng rõ,
đậm và giống nhau hơn. $s = 9$ đẩy mã ra xa vùng của dữ liệu (độ lệch chuẩn 1,58 so với ≈ 1): nét dày, méo, và precision
sụp. Kết quả quét của demo (mạng H = 512, 3 seed, 1.000 mẫu mỗi chữ số, độ đúng do LeNet đọc trên ảnh):

In [ ]:
SCALES = sorted({c["s"] for c in MN["cells"] if c["model"] == "ae16-cfg"})
print(f"{'s':>4} │ đúng chữ số │ precision │ recall    │ FD")
for s_g in SCALES:
    acc_s = cells("ae16-cfg", "cfg", 32, "acc", s=s_g)
    print(f"{vn_tick(s_g):>4} │ {span(acc_s, 2, 100, '%') if acc_s else '—':>11} │ "
          f"{span(cells('ae16-cfg', 'cfg', 32, 'P', s=s_g), 2):>9} │ {span(cells('ae16-cfg', 'cfg', 32, 'R', s=s_g), 2):>9} │ "
          f"{span(cells('ae16-cfg', 'cfg', 32, 'fd', s=s_g))}")  # fmt: skip
print(f"mạng không điều kiện (mục 10), 32 bước: FD {span(cells('ae16-fm', 'euler', 32))}, "
      f"recall {span(cells('ae16-fm', 'euler', 32, 'R'), 2)}")  # fmt: skip

fig, ax = plt.subplots(figsize=(7, 3.4))
for key, color, marker, ls, label in (("acc", BLUE, "o", "-", "đúng chữ số yêu cầu"), ("R", ORANGE, "s", "-", "recall"),
                                      ("P", GREEN, "D", "--", "precision")):  # fmt: skip
    ws = [w for w in SCALES if w > 0]
    vals = [cells("ae16-cfg", "cfg", 32, key, s=w) for w in ws]
    ax.plot(ws, [np.median(v) for v in vals], color=color, ls=ls, lw=1.8, marker=marker, ms=5, label=label)
    for w, v in zip(ws, vals):
        ax.scatter([w] * len(v), v, color=color, marker=marker, s=18, zorder=3)
ax.set_xscale("log")
ax.set(
    xlabel="hệ số hướng dẫn s (thang log)",
    ylabel="tỉ lệ",
    ylim=(0, 1.02),
    title="Hướng dẫn: bám nhãn hơn, kém đa dạng hơn",
)
ax.legend(frameon=False, loc="lower left")
vn_axes(fig)
plt.show()

Đường xanh (chấm tròn) là độ đúng, đường cam (ô vuông) là recall, đường xanh lá đứt (hình thoi) là precision; mỗi dấu
là một seed. Đọc bảng và hình:
- **Điều kiện chưa phải là hướng dẫn.** Mạng có điều kiện ($s = 1$) đã sinh đúng chữ số được yêu cầu 96,48–96,67% số lần,
  và FD 1,8–1,9 đã tốt hơn mạng không điều kiện (3,7–3,9 ở cùng 32 bước).
- **Một chút hướng dẫn giúp, nhiều thì hại.** FD tốt nhất ở $s = 1{,}1$ (1,3–1,4), rồi tăng dần. Ho & Salimans (2022), viết
  $w = s - 1$, cũng thấy FID tốt nhất với $w$ = 0,1 hoặc 0,3.
- **Đổi đa dạng lấy độ bám.** Với $s = 3$, độ đúng lên 99,99%, nhưng recall giảm từ 0,75–0,76 xuống 0,39–0,41 và FD tăng
  lên 8,3–9,0: mẫu nào cũng là một chữ số "điển hình", và những kiểu viết hiếm biến mất.
- **Quá tay thì hỏng cả độ đúng.** Precision lên tới 0,94–0,95 ở $s$ = 2–3 rồi giảm, còn 0,74–0,82 ở $s = 9$, nơi mã đã
  lệch hẳn khỏi vùng của dữ liệu và độ đúng cũng bắt đầu giảm (97,97–99,77%).

Mô hình ảnh thật thường dùng hệ số lớn hơn: pipeline Stable Diffusion 1.x mặc định 7,5, phần lớn thí nghiệm của bài báo
Stable Diffusion 3 chạy ở 5, ví dụ của FLUX.1 [dev] dùng 3,5 (mạng đó được chưng cất hướng dẫn, nên mỗi bước chỉ tốn một
lần gọi mạng).

## 13 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Cụm hẹp hơn thì cần nhiều bước hơn
Đổi độ rộng mỗi cụm của vòng từ 0,1 xuống 0,02 (dữ liệu "sắc" hơn), vẫn với bộ khử nhiễu tối ưu. Đổi `S_TRY` hoặc
`STEPS_TRY` rồi chạy lại.

In [ ]:
S_TRY, STEPS_TRY = 0.02, (2, 4, 8, 32)
for k in STEPS_TRY:
    q_eu = [ring_quality(euler(lambda x, t: ring_v(x, t, S_TRY), z, grid(k)), S_TRY)[0] for z in Z_DEMO]
    q_he = [
        ring_quality(heun(lambda x, t: ring_v(x, t, S_TRY), z, grid(max(1, (k + 1) // 2))), S_TRY)[0] for z in Z_DEMO
    ]
    print(f"{k:>2} lần gọi: Euler {span(q_eu, 1, 100, '%'):>12} · Heun ({max(1, (k + 1) // 2)} bước, "
          f"{2 * max(1, (k + 1) // 2) - 1} lần gọi) {span(q_he, 1, 100, '%')}")  # fmt: skip
    if S_TRY == 0.02:
        assert np.allclose(q_eu, [r["inMode"] for r in RING2D["table"]["0.02"]["rows"][f"euler/{k}"]], atol=1e-3)

Với cụm rộng 0,02, 4 bước Euler chỉ còn 57,5–59,6% mẫu trong một cụm (so với 76,8–78,4% ở độ rộng 0,1): dữ liệu càng
sắc, trường vận tốc càng đổi hướng gắt ở cuối đường, và bước to càng dễ trượt khỏi cụm. Heun, bộ giải bậc hai, ở đây
lại **kém** Euler khi cùng số lần gọi mạng ít (89,9–91,8% với 7 lần gọi, so với 98,2–98,7% của 8 bước Euler): bộ giải
bậc cao chỉ có lợi khi mỗi bước đủ nhỏ để trường gần như trơn trong bước đó. Trên mã MNIST, nơi trường trơn hơn, Heun
lại thắng (mục 10: 9 lần gọi cho FD 2,9–3,2).

### Hướng dẫn cho chữ số khác
Đổi `DIGIT` và `S_LIST`. Ô dưới in độ đúng (bộ phân loại 5 láng giềng của mục 10) và độ trải của mã trong 200 mẫu.

In [ ]:
DIGIT, S_LIST = 8, (1, 2, 4)
z200, y200 = Z_NOISE[2000:2200], np.full(200, DIGIT)
real_spread = spread(Zc[y_img == DIGIT])
for s_g in S_LIST:
    Zd = euler(v_cfg(y200, s_g), z200, grid(32))
    print(f"chữ số {DIGIT}, s = {vn_tick(s_g)}: đúng {pct((knn_digit(Zd) == DIGIT).mean(), 1)}, độ trải của mã "
          f"{vn(spread(Zd))} (mã thật của chữ số {DIGIT}: {vn(real_spread)})")  # fmt: skip

Khi $s$ tăng, độ trải của mã các mẫu thường co lại so với mã thật của cùng chữ số: mẫu giống nhau hơn. Thử $s = 0{,}5$
(giữa không nhãn và có nhãn) để thấy chiều ngược lại.

### Thêm dữ liệu cho mạng tự học
Ở mục 11, đổi `(10, 500)` thành `(10, 50, 100, 500)` để thấy tỉ lệ sao chép giảm dần theo số mã huấn luyện. Ở mục 6, đổi
`train_fm(X_ring)` thành `train_fm(X_ring, steps=1000)` rồi chạy lại mục 6: mạng học kém hơn thì cần nhiều bước Euler hơn
mới vào cụm.

## 14 · Giới hạn và bước tiếp theo

- **Mô hình thu nhỏ.** Ở đây mạng khử nhiễu là một MLP trên 16 số. Stable Diffusion khử nhiễu một lưới mã 64×64×4 (SD3:
  16 kênh) bằng U-Net, rồi bằng Transformer (DiT, MM-DiT của SD3, FLUX.1 với 12 tỉ tham số), và đọc câu mô tả qua
  attention (Bài 16–17) thay cho one-hot của nhãn.
- **Vẫn cần nhiều lần gọi mạng.** 32 bước là ít so với 1.000, nhưng vẫn là 32 lần chạy cả mạng cho một ảnh (64 với
  hướng dẫn). Reflow và chưng cất (consistency models, SDXL Turbo, FLUX.1 [schnell]) đưa con số xuống 1–4.
- **Thước đo của khóa học.** FD ở đây tính trên đặc trưng của LeNet (Bài 12), nên chỉ so được trong khóa học, không so
  được với FID của các bài báo.
- **Học thuộc.** Mục 11 cho thấy cực tiểu của mất mát là chép; mạng thật chỉ khái quát hóa khi đủ dữ liệu.

Câu chữ mà Stable Diffusion đọc đi qua bộ mã hóa văn bản của CLIP, một mô hình **tiền huấn luyện** trên hàng trăm triệu
cặp ảnh–chú thích, và người dùng thường **tinh chỉnh** cả mô hình khuếch tán bằng LoRA để dạy nó một phong cách mới. Đó là
nội dung của bài tiếp theo.

## Tóm tắt
- Quá trình thuận không cần học: $x_t = (1 - t)\,x_0 + t\,\varepsilon$, lấy được ngay ở mọi $t$. Mạng học vận tốc
  $\varepsilon - x_0$ bằng sai số bình phương; mất mát không về 0 (sàn 3,59 trên vòng 8 cụm), vì $x_t$ không cho biết nó đến
  từ cặp nào.
- Đoán $\varepsilon$, đoán $x_0$ và đoán $v$ đổi được cho nhau ($\hat x_0 = x_t - t\,v$), chỉ khác trọng số theo $t$; mỗi
  cách có điểm mù riêng (mạng đoán $\varepsilon$ sai 92–97 ở $t = 0{,}98$, so với 12–13).
- Một bước DDIM là một bước Euler; DDPM, DDIM và flow matching khác nhau ở chỗ đặt các bước. Với bộ khử nhiễu tối ưu và
  4 bước: lưới đều theo $t$ cho 76,8–78,4% mẫu trong một cụm, lưới của DDPM 1,8–1,9%.
- Số bước là lựa chọn của bộ giải: 32 bước Euler cho FD 3,7–3,9, 1.000 bước 3,45. Một bước rơi về trung bình
  (FD 247–284); reflow làm thẳng đường đi, một bước khi ấy cho 5,4–5,6.
- Khuếch tán trong mã của autoencoder Bài 19: FD 3,4–3,6, so với 75–94 khi giải mã nhiễu $\mathcal N(0, I)$ và 7,9–8,0
  của GAN Bài 20.
- Bộ khử nhiễu tối ưu cho một tập hữu hạn chỉ chép (100% bản sao); mạng thật chép 96,2–97,9% khi học trên 100 mã, 0,1–0,2%
  khi học trên 55.000.
- Hướng dẫn không cần bộ phân loại: $s = 3$ cho 99,99% đúng chữ số, nhưng recall 0,75–0,76 → 0,39–0,41.

## Câu hỏi ôn tập
1. Vì sao một bước Euler từ $t = 1$, kể cả với bộ khử nhiễu hoàn hảo, đưa mọi mẫu về cùng một điểm? Điểm đó là gì?
2. Mất mát flow matching của mạng không về 0 dù mạng hoàn hảo. Vì sao, và điều đó nói gì về việc so mất mát của hai mô
   hình?
3. Một mạng đoán $\varepsilon$ và một mạng đoán $v$ học cùng một bộ khử nhiễu. Vì sao mạng đoán $\varepsilon$ lại cho ảnh tệ
   hơn nhiều sau 100 bước Euler?
4. DDPM huấn luyện với $T = 1.000$. Có phải lấy mẫu cũng cần 1.000 bước không? Dẫn hai con số từ notebook.
5. Bộ khử nhiễu tối ưu cho 55.000 mã huấn luyện tính được chính xác. Vì sao người ta vẫn huấn luyện một mạng?
6. Tăng hệ số hướng dẫn $s$ từ 1 lên 3 làm độ đúng tăng và recall giảm. Hai con số đó đo gì, và vì sao chúng đi ngược nhau?

<details><summary>Gợi ý đáp án</summary>

1. Ở $t = 1$, $x_t$ là nhiễu thuần, không mang thông tin gì về $x_0$, nên $\mathbb E[x_0 \mid x_1]$ là trung bình của mọi dữ
   liệu. Bước Euler cỡ 1 đi thẳng tới $\hat x_0$ đó: gốc tọa độ với vòng 8 cụm, "chữ số trung bình" với MNIST.
2. Nhiều cặp $(x_0, \varepsilon)$ cho cùng một $x_t$; mạng chỉ đoán được trung bình của các mục tiêu $\varepsilon - x_0$, phần
   còn lại là sai số không tránh được (3,59 trên vòng 8 cụm). Mức sàn đó phụ thuộc vào dữ liệu và cách chọn $t$, nên mất mát
   không cho biết mô hình nào sinh ảnh tốt hơn.
3. Đổi $\hat\varepsilon$ ra vận tốc phải chia cho $1 - t$; gần $t = 1$, sai số nhỏ của $\hat\varepsilon$ thành sai số vận tốc
   rất lớn (92–97 ở $t = 0{,}98$), đúng ở những bước đầu tiên quyết định chữ số.
4. Không. Ho và cộng sự chọn 1.000 để khớp các công trình trước, không dò tìm. 32 bước Euler cho FD 3,7–3,9 so với 3,45 ở
   1.000 bước; DDIM trên mạng DDPM 100 bước cho 4,1–4,3.
5. Vì nó chỉ chép lại ảnh huấn luyện (100% bản sao). Ta muốn ảnh mới; mạng sinh được ảnh mới chính vì nó không đạt cực
   tiểu đó, nhờ thiên kiến quy nạp và đủ dữ liệu.
6. Độ đúng đo mẫu có đúng chữ số được yêu cầu không (độ bám); recall đo mẫu phủ được bao nhiêu kiểu viết thật (độ đa
   dạng). Hướng dẫn đẩy mỗi mẫu về phía "điển hình nhất" của nhãn, nên mẫu đúng hơn nhưng giống nhau hơn.
</details>

## Tìm hiểu thêm
- J. Ho, A. Jain & P. Abbeel (2020), [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239).
- J. Song, C. Meng & S. Ermon (2020), [DDIM](https://arxiv.org/abs/2010.02502): lấy mẫu tất định, ít bước, cùng mạng DDPM.
- Y. Lipman và cộng sự (2022), [Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747); X. Liu, C. Gong &
  Q. Liu (2022), [Rectified Flow và reflow](https://arxiv.org/abs/2209.03003).
- [Diffusion Meets Flow Matching: Two Sides of the Same Coin](https://diffusionflow.github.io/) (blog của Google DeepMind,
  ICLR 2025): DDIM = Euler của flow matching, và vì sao đường "thẳng" chỉ thẳng cho từng cặp.
- D. Kingma & R. Gao (2023), [Understanding Diffusion Objectives](https://arxiv.org/abs/2303.00848): ε, x₀, v chỉ khác
  trọng số.
- T. Karras và cộng sự (2022), [EDM: Elucidating the Design Space of Diffusion-Based Generative Models](https://arxiv.org/abs/2206.00364).
- R. Rombach và cộng sự (2022), [Latent Diffusion](https://arxiv.org/abs/2112.10752); P. Esser và cộng sự (2024),
  [Stable Diffusion 3](https://arxiv.org/abs/2403.03206); W. Peebles & S. Xie (2022), [DiT](https://arxiv.org/abs/2212.09748).
- J. Ho & T. Salimans (2022), [Classifier-Free Diffusion Guidance](https://arxiv.org/abs/2207.12598).
- N. Carlini và cộng sự (2023), [Extracting Training Data from Diffusion Models](https://arxiv.org/abs/2301.13188);
  Z. Kadkhodaie và cộng sự (2023), [Generalization in diffusion models arises from geometry-adaptive harmonic representations](https://arxiv.org/abs/2310.02557).
- S. Dieleman (2024), [Diffusion is spectral autoregression](https://sander.ai/2024/09/02/spectral-autoregression.html).
- P. Holderrieth & E. Erives (2025), [MIT 6.S184: An Introduction to Flow Matching and Diffusion Models](https://arxiv.org/abs/2506.02070).

**Bài tiếp theo:** [Bài 22 · Tiền huấn luyện & tinh chỉnh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/tien-huan-luyen/):
học trước trên dữ liệu không nhãn, rồi chỉnh một chút cho việc cần làm.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")